# MIMIC-III Clinical Data Analysis
**Beth Israel Deaconess Medical Center · ICU De-identified Demo Dataset**

This notebook covers the full pipeline:
- **Stage 0** — Data loading & validation
- **Stage 1a** — EDA Dashboard (6 charts)
- **Stage 1b** — Entity Relationship Diagram (reference)
- **Stage 2-FE** — Feature engineering (Track A & B)
- **Stage 2A** — Mortality prediction (classification)
- **Stage 2B** — ICU LOS prediction (regression)
- **Stage 2-DB** — Modeling dashboard
- **Stage 3a** — eDISH pharmacovigilance
- **Stage 3b** — Patient story analysis

> **Note:** All HTML outputs are saved as standalone files and can be opened in any browser.


## Setup — Common Config

In [25]:
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/Users/pickle/.cache/kagglehub/datasets/asjad99/mimiciii/versions/1/mimic-iii-clinical-database-demo-1.4"

# Verify path
import os
assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}"
print("✅ DATA_DIR OK:", DATA_DIR)


✅ DATA_DIR OK: /Users/pickle/.cache/kagglehub/datasets/asjad99/mimiciii/versions/1/mimic-iii-clinical-database-demo-1.4


---
## Stage 0 — Data Loading & Validation

In [26]:
"""
阶段 0：数据加载与验证
运行：python stage0_load.py
预期耗时：< 2 min
"""

import pandas as pd
import os

DATA_DIR = "/Users/pickle/.cache/kagglehub/datasets/asjad99/mimiciii/versions/1/mimic-iii-clinical-database-demo-1.4"

# ── 核心表清单 ──────────────────────────────────────────
CORE_TABLES = {
    "PATIENTS":       ("PATIENTS.csv",       ["subject_id", "gender", "dob", "dod"]),
    "ADMISSIONS":     ("ADMISSIONS.csv",     ["subject_id", "hadm_id", "admittime", "dischtime",
                                               "admission_type", "hospital_expire_flag"]),
    "ICUSTAYS":       ("ICUSTAYS.csv",       ["subject_id", "hadm_id", "icustay_id",
                                               "first_careunit", "intime", "outtime", "los"]),
    "DIAGNOSES_ICD":  ("DIAGNOSES_ICD.csv",  ["subject_id", "hadm_id", "icd9_code"]),
    "D_ICD_DIAGNOSES":("D_ICD_DIAGNOSES.csv",["icd9_code", "long_title"]),
    "LABEVENTS":      ("LABEVENTS.csv",      ["subject_id", "hadm_id", "itemid", "charttime", "valuenum"]),
    "CHARTEVENTS":    ("CHARTEVENTS.csv",    ["subject_id", "hadm_id", "icustay_id",
                                               "itemid", "charttime", "valuenum", "error"]),
    "PRESCRIPTIONS":  ("PRESCRIPTIONS.csv",  ["subject_id", "hadm_id", "drug"]),
}

# Lab 指标 itemid（用于 LABEVENTS 快速过滤）
LAB_ITEMIDS = [50861, 50885, 50912, 50813, 51006, 51222, 51265, 51301]

# 体征 itemid（用于 CHARTEVENTS 快速过滤）
VITAL_ITEMIDS = [211, 220045, 51, 220050, 8368, 220051, 646, 220277, 223761, 678, 618, 220210]


def load_table(name, filename, usecols=None, parse_dates=None):
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        print(f"  ✗ 文件不存在：{path}")
        return None
    # 只加载需要的列（节省内存）
    df = pd.read_csv(path, usecols=usecols, parse_dates=parse_dates, low_memory=False)
    print(f"  ✓ {name:<20} {len(df):>8,} 行  列：{list(df.columns)}")
    return df


def load_chartevents_filtered(itemids):
    """CHARTEVENTS 体积大，按 itemid 分块过滤后合并，避免 OOM。"""
    path = os.path.join(DATA_DIR, "CHARTEVENTS.csv")
    if not os.path.exists(path):
        print(f"  ✗ 文件不存在：{path}")
        return None

    usecols = ["subject_id", "hadm_id", "icustay_id", "itemid", "charttime", "valuenum", "error"]
    chunks = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=100_000,
                              parse_dates=["charttime"], low_memory=False):
        filtered = chunk[chunk["itemid"].isin(itemids)]
        if len(filtered):
            chunks.append(filtered)

    if not chunks:
        print("  ✗ CHARTEVENTS：过滤后无数据")
        return None

    df = pd.concat(chunks, ignore_index=True)
    # 排除错误记录
    if "error" in df.columns:
        df = df[df["error"].isna() | (df["error"] == 0)]
    print(f"  ✓ CHARTEVENTS (filtered)    {len(df):>8,} 行  (itemids={itemids})")
    return df


def validate_admissions(df):
    """核查 ADMISSIONS 关键字段。"""
    print("\n── ADMISSIONS 健康检查 ──")
    n_total    = len(df)
    n_null_flag = df["hospital_expire_flag"].isna().sum()
    # 陷阱：hospital_expire_flag 有 NULL，必须 fillna(0) 后再 sum，否则死亡率偏低
    flag_clean = df["hospital_expire_flag"].fillna(0).astype(int)
    n_expired  = flag_clean.sum()
    print(f"  总住院次数：{n_total}")
    print(f"  hospital_expire_flag NULL：{n_null_flag}  (已填充为 0)")
    print(f"  院内死亡：{n_expired}  ({n_expired/n_total*100:.1f}%)")
    print(f"  入院类型分布：\n{df['admission_type'].value_counts().to_string()}")


def validate_icustays(df):
    """核查 ICUSTAYS。"""
    print("\n── ICUSTAYS 健康检查 ──")
    print(f"  ICU 住院次数：{len(df)}")
    print(f"  LOS（天）中位数：{df['los'].median():.2f}")
    print(f"  护理单元分布：\n{df['first_careunit'].value_counts().to_string()}")


def main():
    print("=" * 55)
    print("阶段 0：数据加载验证")
    print("=" * 55)

    dfs = {}

    # ── 加载小表 ──
    print("\n[1] 加载核心小表 ...")
    for name, (fname, cols) in CORE_TABLES.items():
        if name == "CHARTEVENTS":
            continue  # 大表单独处理
        parse_dates = None
        if name == "PATIENTS":
            parse_dates = ["dob", "dod"]
        elif name == "ADMISSIONS":
            parse_dates = ["admittime", "dischtime"]
        elif name == "ICUSTAYS":
            parse_dates = ["intime", "outtime"]
        elif name == "LABEVENTS":
            parse_dates = ["charttime"]

        dfs[name] = load_table(name, fname, usecols=cols, parse_dates=parse_dates)

    # ── 加载 LABEVENTS（中等大小，全量加载可接受）──
    print("\n[2] 加载 LABEVENTS（全量）...")
    lab_path = os.path.join(DATA_DIR, "LABEVENTS.csv")
    if os.path.exists(lab_path):
        dfs["LABEVENTS"] = pd.read_csv(
            lab_path,
            usecols=["subject_id", "hadm_id", "itemid", "charttime", "valuenum"],
            parse_dates=["charttime"],
            low_memory=False,
        )
        print(f"  ✓ LABEVENTS              {len(dfs['LABEVENTS']):>8,} 行")

    # ── 加载 CHARTEVENTS（分块过滤）──
    print("\n[3] 加载 CHARTEVENTS（分块过滤，仅保留体征 itemid）...")
    dfs["CHARTEVENTS"] = load_chartevents_filtered(VITAL_ITEMIDS)

    # ── 健康检查 ──
    print("\n[4] 核心表健康检查 ...")
    if dfs.get("ADMISSIONS") is not None:
        validate_admissions(dfs["ADMISSIONS"])
    if dfs.get("ICUSTAYS") is not None:
        validate_icustays(dfs["ICUSTAYS"])

    print("\n✅ 阶段 0 完成，数据已就绪，继续执行 stage1a_eda.py")
    return dfs


if __name__ == "__main__":
    main()

阶段 0：数据加载验证

[1] 加载核心小表 ...
  ✓ PATIENTS                  100 行  列：['subject_id', 'gender', 'dob', 'dod']
  ✓ ADMISSIONS                129 行  列：['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type', 'hospital_expire_flag']
  ✓ ICUSTAYS                  136 行  列：['subject_id', 'hadm_id', 'icustay_id', 'first_careunit', 'intime', 'outtime', 'los']
  ✓ DIAGNOSES_ICD           1,761 行  列：['subject_id', 'hadm_id', 'icd9_code']
  ✓ D_ICD_DIAGNOSES        14,567 行  列：['icd9_code', 'long_title']
  ✓ LABEVENTS              76,074 行  列：['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum']
  ✓ PRESCRIPTIONS          10,398 行  列：['subject_id', 'hadm_id', 'drug']

[2] 加载 LABEVENTS（全量）...
  ✓ LABEVENTS                76,074 行

[3] 加载 CHARTEVENTS（分块过滤，仅保留体征 itemid）...
  ✓ CHARTEVENTS (filtered)      64,588 行  (itemids=[211, 220045, 51, 220050, 8368, 220051, 646, 220277, 223761, 678, 618, 220210])

[4] 核心表健康检查 ...

── ADMISSIONS 健康检查 ──
  总住院次数：129
  hospital_expire_flag NULL：0 

---
## Stage 1a — EDA Dashboard

Generates `eda_dashboard.html`. Open in browser after running.

In [43]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os

# DATA_DIR is set in the Setup cell above
OUTPUT_FILE = "eda_dashboard.html"

# 全局白底主题
LAYOUT_BASE = dict(
    paper_bgcolor="#ffffff",
    plot_bgcolor="#f8f9fa",
    font=dict(color="#1a1a2e", family="Arial, sans-serif"),
)

ACCENT  = ["#2563EB", "#10B981", "#F59E0B", "#EF4444", "#8B5CF6", "#EC4899"]
C_DEAD  = "#EF4444"
C_ALIVE = "#10B981"

def hex_to_rgba(hex_color, alpha=0.25):
    """Convert #RRGGBB to rgba(r,g,b,alpha) for Plotly fillcolor."""
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha})"


# ════════════════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════════════════

def load_data():
    def read(fname, cols=None, dates=None):
        path = os.path.join(DATA_DIR, fname)
        return pd.read_csv(path, usecols=cols, parse_dates=dates, low_memory=False)

    admissions = read("ADMISSIONS.csv",
        cols=["subject_id","hadm_id","admittime","dischtime",
              "admission_type","hospital_expire_flag"],
        dates=["admittime","dischtime"])
    icustays = read("ICUSTAYS.csv",
        cols=["subject_id","hadm_id","icustay_id","first_careunit","los"])
    diagnoses = read("DIAGNOSES_ICD.csv",
        cols=["subject_id","hadm_id","icd9_code"])
    d_icd = read("D_ICD_DIAGNOSES.csv",
        cols=["icd9_code","long_title"])
    labevents = read("LABEVENTS.csv",
        cols=["subject_id","hadm_id","itemid","charttime","valuenum"],
        dates=["charttime"])

    admissions["hospital_expire_flag"] = (
        admissions["hospital_expire_flag"].fillna(0).astype(int))
    return admissions, icustays, diagnoses, d_icd, labevents


# ════════════════════════════════════════════════════════
# 图 1：In-Hospital Mortality Rate
# ════════════════════════════════════════════════════════

def fig_mortality(admissions):
    n_total = len(admissions)
    n_dead  = int(admissions["hospital_expire_flag"].sum())
    n_alive = n_total - n_dead
    rate    = n_dead / n_total * 100

    fig = go.Figure(go.Pie(
        labels=["Survived", "In-Hospital Death"],
        values=[n_alive, n_dead],
        hole=0.58,
        marker=dict(colors=[C_ALIVE, C_DEAD],
                    line=dict(color="#ffffff", width=2)),
        textinfo="label+percent",
        textfont=dict(size=13, color="#1a1a2e"),
        hovertemplate="%{label}: %{value} patients (%{percent})<extra></extra>",
        direction="clockwise",
    ))
    # 中心标注
    fig.add_annotation(
        text=f"<b>{rate:.1f}%</b><br><span style='font-size:11px'>Mortality</span>",
        x=0.5, y=0.5, showarrow=False,
        font=dict(size=20, color="#1a1a2e"),
        xanchor="center", yanchor="middle",
    )
    # 底部统计行 — 用两行分开，行距充足
    fig.add_annotation(
        text=f"Total Admissions: <b>{n_total}</b>",
        x=0.5, y=-0.10, showarrow=False,
        font=dict(size=12, color="#444"),
        xref="paper", yref="paper", xanchor="center",
    )
    fig.add_annotation(
        text=f"Deaths: <b>{n_dead}</b> &nbsp;|&nbsp; Survived: <b>{n_alive}</b>",
        x=0.5, y=-0.18, showarrow=False,
        font=dict(size=12, color="#444"),
        xref="paper", yref="paper", xanchor="center",
    )
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="In-Hospital Mortality Rate",
                   font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"),
        showlegend=True,
        legend=dict(orientation="h", y=-0.28, x=0.5, xanchor="center",
                    font=dict(size=12)),
        margin=dict(t=50, b=110, l=20, r=20),
        height=400,
    )
    return fig


# ════════════════════════════════════════════════════════
# 图 2：Top 10 Most Common Diagnoses
# ════════════════════════════════════════════════════════

def fig_top_diagnoses(diagnoses, d_icd):
    merged = diagnoses.merge(d_icd, on="icd9_code", how="left")
    merged["label"] = merged["long_title"].fillna(merged["icd9_code"].astype(str))

    top10 = (
        merged.groupby("label")["hadm_id"].count()
        .nlargest(10).reset_index()
        .rename(columns={"hadm_id": "count"})
        .sort_values("count")
    )
    def wrap_label(text, width=30):
        words = text.split()
        lines, cur = [], []
        for w in words:
            if sum(len(x)+1 for x in cur) + len(w) > width and cur:
                lines.append(' '.join(cur))
                cur = [w]
            else:
                cur.append(w)
        if cur: lines.append(' '.join(cur))
        return '<br>'.join(lines)
    top10["label_short"] = top10["label"].apply(wrap_label)

    fig = go.Figure(go.Bar(
        x=top10["count"],
        y=top10["label_short"],
        orientation="h",
        marker=dict(
            color=top10["count"],
            colorscale=[[0,"#93C5FD"],[1,"#1D4ED8"]],
            showscale=False,
        ),
        text=top10["count"],
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Count: %{x}<extra></extra>",
    ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="Top 10 Most Common Diagnoses (ICD-9)",
                   font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"),
        xaxis=dict(title="Count", title_font=dict(size=12),
                   showgrid=True, gridcolor="#e5e7eb"),
        yaxis=dict(automargin=True, tickfont=dict(size=10)),
        margin=dict(t=50, b=60, l=20, r=60),   # r=60 给 text 留空间
        height=420,
    )
    return fig


# ════════════════════════════════════════════════════════
# 图 3：ICU Length of Stay by Care Unit
# ════════════════════════════════════════════════════════

# def fig_los_by_careunit(icustays):
#     df = icustays[icustays["los"] > 0].copy()
#     units = df["first_careunit"].value_counts().index.tolist()
#     y_max = min(df["los"].quantile(0.95) * 1.2, 35)

#     fig = go.Figure()
#     for i, unit in enumerate(units):
#         sub = df[df["first_careunit"] == unit]["los"]
#         fig.add_trace(go.Box(
#             y=sub,
#             name=unit,
#             marker=dict(color=ACCENT[i % len(ACCENT)]),
#             line=dict(color=ACCENT[i % len(ACCENT)]),
#             fillcolor=hex_to_rgba(ACCENT[i % len(ACCENT)], 0.25),
#             boxmean="sd",
#             hovertemplate=f"<b>{unit}</b><br>LOS: %{{y:.1f}} days<extra></extra>",
#         ))

#     fig.update_layout(
#         paper_bgcolor="#ffffff",
#         plot_bgcolor="#f8f9fa",
#         font=dict(color="#1a1a2e", family="Arial, sans-serif"),
#         template="none",
#         title=dict(text="ICU Length of Stay by Care Unit",
#                    font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"),
#         yaxis=dict(
#             title="LOS (days)",
#             range=[0, y_max],
#             showgrid=True, gridcolor="#e5e7eb",
#             zeroline=True, zerolinecolor="#cbd5e1",
#             title_font=dict(size=12),
#             tickfont=dict(size=11),
#             showline=True, linecolor="#e5e7eb",
#         ),
#         xaxis=dict(
#             title="Care Unit",
#             title_font=dict(size=12),
#             tickfont=dict(size=11),
#             showline=True, linecolor="#e5e7eb",
#         ),
#         showlegend=False,
#         margin=dict(t=55, b=60, l=65, r=20),
#         height=440,
#         boxgap=0.3,
#         boxgroupgap=0.2,
#     )
#     return fig

def fig_los_by_careunit(icustays):
    df = icustays.copy()
    df["los"] = pd.to_numeric(df["los"], errors="coerce")
    df["first_careunit"] = df["first_careunit"].astype(str).str.strip()

    df = df.dropna(subset=["los", "first_careunit"])
    df = df[(df["los"] > 0) & (df["first_careunit"] != "")]

    print("fig3 rows:", len(df))
    print(df["first_careunit"].value_counts())

    if df.empty:
        fig = go.Figure()
        fig.update_layout(
            **LAYOUT_BASE,
            title="ICU Length of Stay by Care Unit (No valid data)"
        )
        return fig

    units = df["first_careunit"].value_counts().index.tolist()
    y_max = min(df["los"].quantile(0.95) * 1.2, 35)

    fig = go.Figure()
    for i, unit in enumerate(units):
        sub = df.loc[df["first_careunit"] == unit, "los"]
        fig.add_trace(go.Box(
            y=sub,
            name=unit,
            marker=dict(color=ACCENT[i % len(ACCENT)]),
            line=dict(color=ACCENT[i % len(ACCENT)]),
            fillcolor=hex_to_rgba(ACCENT[i % len(ACCENT)], 0.25),
            boxmean="sd",
            hovertemplate=f"<b>{unit}</b><br>LOS: %{{y:.1f}} days<extra></extra>",
        ))

    fig.update_layout(
        paper_bgcolor="#ffffff",
        plot_bgcolor="#f8f9fa",
        font=dict(color="#1a1a2e", family="Arial, sans-serif"),
        template="none",
        title=dict(text="ICU Length of Stay by Care Unit",
                   font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"),
        yaxis=dict(
            title="LOS (days)",
            range=[0, y_max],
            showgrid=True, gridcolor="#e5e7eb",
            zeroline=True, zerolinecolor="#cbd5e1",
            title_font=dict(size=12),
            tickfont=dict(size=11),
            showline=True, linecolor="#e5e7eb",
        ),
        xaxis=dict(
            title="Care Unit",
            title_font=dict(size=12),
            tickfont=dict(size=11),
            showline=True, linecolor="#e5e7eb",
        ),
        showlegend=False,
        margin=dict(t=55, b=60, l=65, r=20),
        height=440,
        boxgap=0.3,
        boxgroupgap=0.2,
    )
    return fig


# ════════════════════════════════════════════════════════
# 图 4：Mortality Rate by Admission Type
# ════════════════════════════════════════════════════════

# def fig_mortality_by_admission_type(admissions):
#     adm = admissions.copy()
#     adm["hospital_expire_flag"] = adm["hospital_expire_flag"].fillna(0).astype(int)
#     grouped = (
#         adm.groupby("admission_type")["hospital_expire_flag"]
#         .agg(total="count", dead="sum").reset_index()
#     )
#     grouped["rate"] = grouped["dead"] / grouped["total"] * 100

#     COLOR_MAP = {"EMERGENCY": "#EF4444", "ELECTIVE": "#10B981", "URGENT": "#F59E0B"}
#     colors = [COLOR_MAP.get(t, "#6B7280") for t in grouped["admission_type"]]

#     fig = go.Figure(go.Bar(
#         x=grouped["admission_type"],
#         y=grouped["rate"],
#         marker_color=colors,
#         marker_line=dict(color="#ffffff", width=1.5),
#         text=[f"{r:.1f}%" for r in grouped["rate"]],
#         textposition="outside",
#         textfont=dict(size=13, color="#1a1a2e"),
#         width=0.45,       # 控制柱宽，不让柱子撑满
#         customdata=grouped[["dead","total"]].values,
#         hovertemplate=(
#             "<b>%{x}</b><br>"
#             "Mortality Rate: %{y:.1f}%<br>"
#             "Deaths: %{customdata[0]} / Total: %{customdata[1]}"
#             "<extra></extra>"
#         ),
#     ))
#     y_max = max(grouped["rate"].max() * 1.35, 10)   # 比最高值多 35%，保证标签不截断
#     fig.update_layout(
#         **LAYOUT_BASE,
#         title=dict(text="Mortality Rate by Admission Type",
#                    font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"),
#         yaxis=dict(title="Mortality Rate (%)", range=[0, y_max],
#                    showgrid=True, gridcolor="#e5e7eb",
#                    title_font=dict(size=12)),
#         xaxis=dict(title="Admission Type", title_font=dict(size=12),
#                    tickfont=dict(size=12)),
#         bargap=0.5,
#         margin=dict(t=50, b=60, l=60, r=20),
#         height=400,
#     )
#     return fig

def fig_mortality_by_admission_type(admissions):
    adm = admissions.copy()
    adm["hospital_expire_flag"] = pd.to_numeric(
        adm["hospital_expire_flag"], errors="coerce"
    ).fillna(0).astype(int)

    adm["admission_type"] = (
        adm["admission_type"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    grouped = (
        adm.groupby("admission_type", dropna=False)["hospital_expire_flag"]
        .agg(total="count", dead="sum")
        .reset_index()
    )
    grouped["rate"] = grouped["dead"] / grouped["total"] * 100

    print(grouped)

    COLOR_MAP = {"EMERGENCY": "#EF4444", "ELECTIVE": "#10B981", "URGENT": "#F59E0B"}
    colors = [COLOR_MAP.get(t, "#6B7280") for t in grouped["admission_type"]]

    fig = go.Figure(go.Bar(
        x=grouped["admission_type"],
        y=grouped["rate"],
        marker_color=colors,
        marker_line=dict(color="#ffffff", width=1.5),
        text=[f"{r:.1f}%" for r in grouped["rate"]],
        textposition="outside",
        textfont=dict(size=13, color="#1a1a2e"),
        customdata=grouped[["dead", "total"]].to_numpy(),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Mortality Rate: %{y:.1f}%<br>"
            "Deaths: %{customdata[0]} / Total: %{customdata[1]}"
            "<extra></extra>"
        ),
    ))

    y_max = max(grouped["rate"].max() * 1.35, 10)
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="Mortality Rate by Admission Type",
                   font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"),
        yaxis=dict(title="Mortality Rate (%)", range=[0, y_max],
                   showgrid=True, gridcolor="#e5e7eb",
                   title_font=dict(size=12)),
        xaxis=dict(title="Admission Type", title_font=dict(size=12),
                   tickfont=dict(size=12)),
        bargap=0.5,
        margin=dict(t=50, b=60, l=60, r=20),
        height=400,
    )
    return fig

# ════════════════════════════════════════════════════════
# 图 5：Lab Value Distributions
# ════════════════════════════════════════════════════════

def fig_lab_distributions(labevents):
    LAB_ITEMS = {
        50912: ("Creatinine", "mg/dL", (0, 15),  1.0),
        51301: ("WBC",        "K/uL",  (0, 50),  2.0),
        51222: ("Hemoglobin", "g/dL",  (0, 20),  0.5),
    }
    COLORS = ["#2563EB", "#EF4444", "#10B981"]

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[v[0] for v in LAB_ITEMS.values()],
        horizontal_spacing=0.12,
    )

    for col_idx, (itemid, (name, unit, clip_range, bin_size)) in enumerate(LAB_ITEMS.items(), 1):
        sub = labevents[labevents["itemid"] == itemid]["valuenum"].dropna()
        sub = sub[(sub > clip_range[0]) & (sub <= clip_range[1])]

        print(f"  {name}: {len(sub)} records")
        if len(sub) == 0:
            fig.add_trace(go.Scatter(x=[], y=[], showlegend=False), row=1, col=col_idx)
            continue

        median_val = sub.median()
        mean_val   = sub.mean()

        fig.add_trace(
            go.Histogram(
                x=sub,
                xbins=dict(start=clip_range[0], end=clip_range[1], size=bin_size),
                marker=dict(
                    color=hex_to_rgba(COLORS[col_idx-1], 0.75),
                    line=dict(color=COLORS[col_idx-1], width=0.8),
                ),
                name=name,
                hovertemplate=f"<b>{name}</b><br>{unit}: %{{x:.1f}}<br>Count: %{{y}}<extra></extra>",
                showlegend=False,
            ),
            row=1, col=col_idx,
        )
        # 中位线 + 均值线
        fig.add_vline(x=median_val, line_color=COLORS[col_idx-1], line_width=2,
                      line_dash="dash", row=1, col=col_idx,
                      annotation_text=f"Median {median_val:.1f}",
                      annotation_font=dict(size=10, color=COLORS[col_idx-1]),
                      annotation_position="top right")
        fig.add_vline(x=mean_val, line_color="#F59E0B", line_width=1.5,
                      line_dash="dot", row=1, col=col_idx)

        fig.update_xaxes(title_text=unit, row=1, col=col_idx,
                         title_font=dict(size=11),
                         showgrid=True, gridcolor="#e5e7eb",
                         showline=True, linecolor="#e5e7eb")
        fig.update_yaxes(title_text="Count" if col_idx == 1 else "",
                         row=1, col=col_idx,
                         showgrid=True, gridcolor="#e5e7eb")

    for ann in fig.layout.annotations:
        ann.font.color = "#1a1a2e"
        ann.font.size  = 13

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="Lab Value Distributions — All Measurements",
                   font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"),
        margin=dict(t=80, b=60, l=60, r=20),
        height=420,
    )
    return fig


# ════════════════════════════════════════════════════════
# 图 6：ICU LOS Distribution
# ════════════════════════════════════════════════════════

def fig_los_distribution(icustays):
    los = icustays[icustays["los"] > 0]["los"]
    median_los = los.median()
    mean_los   = los.mean()
    p75_los    = los.quantile(0.75)

    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=los,
        xbins=dict(start=0, end=los.quantile(0.97), size=0.5),
        marker=dict(color="#8B5CF6", opacity=0.75,
                    line=dict(color="#ffffff", width=0.5)),
        name="ICU LOS",
        hovertemplate="LOS: %{x:.1f} days<br>Count: %{y}<extra></extra>",
    ))
    # 中位数线
    fig.add_vline(x=median_los, line_dash="dash", line_color="#EF4444", line_width=2,
                  annotation_text=f"Median: {median_los:.1f}d",
                  annotation_position="top right",
                  annotation_font=dict(size=11, color="#EF4444"))
    # 均值线
    fig.add_vline(x=mean_los, line_dash="dot", line_color="#F59E0B", line_width=2,
                  annotation_text=f"Mean: {mean_los:.1f}d",
                  annotation_position="top left",
                  annotation_font=dict(size=11, color="#F59E0B"))
    # P75 线
    fig.add_vline(x=p75_los, line_dash="longdash", line_color="#2563EB", line_width=1.5,
                  annotation_text=f"P75: {p75_los:.1f}d",
                  annotation_position="top right",
                  annotation_font=dict(size=11, color="#2563EB"))

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="ICU Length of Stay Distribution",
                   font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"),
        xaxis=dict(title="LOS (days)", showgrid=True, gridcolor="#e5e7eb",
                   title_font=dict(size=12)),
        yaxis=dict(title="Count", showgrid=True, gridcolor="#e5e7eb",
                   title_font=dict(size=12)),
        margin=dict(t=50, b=60, l=60, r=20),
        height=400,
        showlegend=False,
    )
    return fig


# ════════════════════════════════════════════════════════
# 组装 HTML
# ════════════════════════════════════════════════════════

def build_dashboard(figs, output_path):
    divs = []
    for fig in figs.values():
        div = pio.to_html(fig, full_html=False, include_plotlyjs=False)
        divs.append(f'<div class="chart-card">{div}</div>')

    charts_html = "\n".join(divs)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>MIMIC-III EDA Dashboard</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{
      font-family: 'Segoe UI', Arial, sans-serif;
      background: #f1f5f9;
      color: #1a1a2e;
      min-height: 100vh;
    }}
    header {{
      background: #ffffff;
      border-bottom: 2px solid #e2e8f0;
      padding: 24px 40px 18px;
      box-shadow: 0 1px 4px rgba(0,0,0,0.06);
    }}
    header h1 {{
      font-size: 22px;
      font-weight: 700;
      color: #1e293b;
      letter-spacing: 0.3px;
    }}
    header p {{
      font-size: 13px;
      color: #64748b;
      margin-top: 4px;
    }}
    .grid {{
      display: grid;
      grid-template-columns: repeat(2, 1fr);
      gap: 20px;
      padding: 28px 40px;
      max-width: 1440px;
      margin: 0 auto;
    }}
    .chart-card {{
      background: #ffffff;
      border: 1px solid #e2e8f0;
      border-radius: 12px;
      padding: 20px;
      box-shadow: 0 1px 6px rgba(0,0,0,0.06);
      transition: box-shadow 0.2s, border-color 0.2s;
      overflow: hidden;
    }}
    .chart-card:hover {{
      box-shadow: 0 4px 16px rgba(37,99,235,0.10);
      border-color: #93C5FD;
    }}
  </style>
</head>
<body>
  <header>
    <h1>MIMIC-III Clinical Data — EDA Dashboard</h1>
    <p>Level 1a · Exploratory Data Analysis · Beth Israel Deaconess Medical Center ICU (De-identified Demo Subset)</p>
  </header>
  <div class="grid">
    {charts_html}
  </div>
</body>
</html>"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ Dashboard saved: {output_path}")


# ════════════════════════════════════════════════════════
# 主函数
# ════════════════════════════════════════════════════════

def main():
    
    print("=" * 55)
    print("Stage 1a: EDA Dashboard")
    print("=" * 55)

    print("\n[1] Loading data ...")
    admissions, icustays, diagnoses, d_icd, labevents = load_data()
    print(f"  ADMISSIONS:  {len(admissions):,}")
    print(f"  ICUSTAYS:    {len(icustays):,}")
    print(f"  DIAGNOSES:   {len(diagnoses):,}")
    print(f"  LABEVENTS:   {len(labevents):,}")

    fig3 = fig_los_by_careunit(icustays)
    fig4 = fig_mortality_by_admission_type(admissions)

    fig3.write_html("debug_fig3.html", include_plotlyjs="cdn")
    fig4.write_html("debug_fig4.html", include_plotlyjs="cdn")

    print("\n[2] Generating charts ...")
    figs = {
        "mortality":         fig_mortality(admissions),
        "top_diagnoses":     fig_top_diagnoses(diagnoses, d_icd),
        "los_by_careunit":   fig_los_by_careunit(icustays),
        "mortality_by_type": fig_mortality_by_admission_type(admissions),
        "lab_distributions": fig_lab_distributions(labevents),
        "los_distribution":  fig_los_distribution(icustays),
    }
    print(f"  ✓ {len(figs)} charts generated")

    print("\n[3] Exporting dashboard ...")
    build_dashboard(figs, OUTPUT_FILE)
    print(f"\n✅ Done! Open in browser: {OUTPUT_FILE}")



main()


Stage 1a: EDA Dashboard

[1] Loading data ...
  ADMISSIONS:  129
  ICUSTAYS:    136
  DIAGNOSES:   1,761
  LABEVENTS:   76,074
fig3 rows: 136
first_careunit
MICU     77
SICU     23
CCU      19
TSICU    11
CSRU      6
Name: count, dtype: int64
  admission_type  total  dead       rate
0       ELECTIVE      8     0   0.000000
1      EMERGENCY    119    39  32.773109
2         URGENT      2     1  50.000000

[2] Generating charts ...
fig3 rows: 136
first_careunit
MICU     77
SICU     23
CCU      19
TSICU    11
CSRU      6
Name: count, dtype: int64
  admission_type  total  dead       rate
0       ELECTIVE      8     0   0.000000
1      EMERGENCY    119    39  32.773109
2         URGENT      2     1  50.000000
  Creatinine: 2172 records
  WBC: 2015 records
  Hemoglobin: 2023 records
  ✓ 6 charts generated

[3] Exporting dashboard ...
✅ Dashboard saved: eda_dashboard.html

✅ Done! Open in browser: eda_dashboard.html


In [28]:
# Display inline preview of the mortality chart
figs_preview = {}
import pandas as pd, os
from plotly.subplots import make_subplots
import plotly.graph_objects as go
# Quick inline chart — just show figure 1 inline
admissions_tmp = pd.read_csv(os.path.join(DATA_DIR, "ADMISSIONS.csv"),
                              usecols=["hospital_expire_flag","admission_type"])
admissions_tmp["hospital_expire_flag"] = admissions_tmp["hospital_expire_flag"].fillna(0).astype(int)
fig_preview = fig_mortality(admissions_tmp)
fig_preview.show()


---
## Stage 1b — Entity Relationship Diagram

The ERD is in `mimic_iii_erd.drawio` — open with [app.diagrams.net](https://app.diagrams.net).

Key relationships:
```
PATIENTS (subject_id PK)
  └─► ADMISSIONS (hadm_id PK, subject_id FK)
        └─► ICUSTAYS (icustay_id PK, hadm_id FK)
        └─► LABEVENTS (subject_id FK, hadm_id FK, itemid FK)
        └─► DIAGNOSES_ICD (subject_id FK, hadm_id FK, icd9_code FK)
        └─► PRESCRIPTIONS (subject_id FK, hadm_id FK)
              ICUSTAYS └─► CHARTEVENTS (icustay_id FK, itemid FK)
```


---
## Stage 2-FE — Feature Engineering

Outputs: `features_track_a.csv` and `features_track_b.csv`

**Key design decisions:**
- Age: patients with `dob < 1900` have been shifted by MIMIC for privacy. We assign `age=91` (sentinel) and `age_imputed=1`
- Charlson index: hand-coded ICD-9 prefix mapping (Quan 2005)
- Missing values: filled with global median per feature
- CHARTEVENTS: chunked loading to avoid OOM

In [29]:
import pandas as pd
import numpy as np
import os

# DATA_DIR is set in the Setup cell above


# ════════════════════════════════════════════════════════
# 工具函数
# ════════════════════════════════════════════════════════

def read(fname, cols=None, dates=None):
    path = os.path.join(DATA_DIR, fname)
    return pd.read_csv(path, usecols=cols, parse_dates=dates, low_memory=False)


def load_chartevents_filtered(itemids):
    """CHARTEVENTS 大文件，分块过滤后合并。"""
    path = os.path.join(DATA_DIR, "CHARTEVENTS.csv")
    usecols = ["subject_id", "hadm_id", "icustay_id", "itemid", "charttime", "valuenum", "error"]
    chunks = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=100_000,
                              parse_dates=["charttime"], low_memory=False):
        sub = chunk[chunk["itemid"].isin(itemids)].copy()
        # 排除 error 记录
        if "error" in sub.columns:
            sub = sub[sub["error"].isna() | (sub["error"] == 0)]
        if len(sub):
            chunks.append(sub)
    df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
    print(f"  CHARTEVENTS filtered: {len(df):,} 行")
    return df


# ════════════════════════════════════════════════════════
# 共用特征 1：人口统计学
# ════════════════════════════════════════════════════════

def build_demographics(patients, admissions):
    """
    返回以 hadm_id 为索引的人口统计特征 DataFrame：
      age, age_imputed, gender_M, adm_EMERGENCY, adm_ELECTIVE, adm_URGENT

    age_imputed = 1 表示该患者年龄 >89 岁，dob 被 MIMIC 位移无法精确计算。
    age 填入哨兵值 91，在数值上明确大于 89，保留「高龄」方向信息，
    同时由 age_imputed 标志让模型单独学习该组行为。
    """
    df = admissions[["hadm_id", "subject_id", "admittime",
                      "dischtime", "admission_type", "hospital_expire_flag"]].copy()
    df = df.merge(patients[["subject_id", "dob", "gender"]], on="subject_id", how="left")

    df["admittime"] = pd.to_datetime(df["admittime"])
    df["dob"]       = pd.to_datetime(df["dob"])

    # MIMIC 对 >89 岁患者将 dob 位移至 1800 年代，正常患者 dob >= 1900
    normal_mask = df["dob"] >= pd.Timestamp("1900-01-01")

    df["age"] = np.nan
    df["age_imputed"] = 0  # 0 = 真实年龄，1 = 年龄未知(>89岁，哨兵值 91)

    # 正常患者：精确计算年龄
    df.loc[normal_mask, "age"] = (
        df.loc[normal_mask, "admittime"] - df.loc[normal_mask, "dob"]
    ).dt.days / 365.25

    # >89 岁患者：填哨兵值 91（数值上明确 >89，保留高龄方向信息）+ 打标记
    df.loc[~normal_mask, "age"] = 91.0
    df.loc[~normal_mask, "age_imputed"] = 1

    n_imputed = (~normal_mask).sum()
    print(f"    年龄：正常 {normal_mask.sum()} 条，>89岁哨兵填充 {n_imputed} 条（age=91）")

    # 性别 One-hot
    df["gender_M"] = (df["gender"] == "M").astype(int)

    # 入院类型 One-hot
    for atype in ["EMERGENCY", "ELECTIVE", "URGENT"]:
        df[f"adm_{atype}"] = (df["admission_type"] == atype).astype(int)

    cols = ["hadm_id", "subject_id", "admittime", "dischtime",
            "hospital_expire_flag", "age", "age_imputed", "gender_M",
            "adm_EMERGENCY", "adm_ELECTIVE", "adm_URGENT"]
    return df[cols]


# ════════════════════════════════════════════════════════
# 共用特征 2：Charlson 合并症指数
# ════════════════════════════════════════════════════════

# ICD-9 前缀 → Charlson 疾病分类及权重
# 参考：Quan et al. 2005 更新版
CHARLSON_MAP = {
    # weight: 1
    "MI":          (1, ["410", "412"]),
    "CHF":         (1, ["428"]),
    "PVD":         (1, ["440", "441", "4431", "4432", "4438", "4439", "4471", "5571", "5579", "V434"]),
    "CVD":         (1, ["43", "4379"]),
    "Dementia":    (1, ["290"]),
    "COPD":        (1, ["490", "491", "492", "493", "494", "495", "496", "500", "501", "502", "503", "504", "505"]),
    "Rheumatic":   (1, ["4465", "7100", "7101", "7102", "7103", "7104", "7140", "7141", "7142", "7148"]),
    "PUD":         (1, ["531", "532", "533", "534"]),
    "MildLiver":   (1, ["5712", "5714", "5715", "5716"]),
    "DiabNoComp":  (1, ["2500", "2501", "2502", "2503", "2508", "2509"]),
    # weight: 2
    "DiabComp":    (2, ["2504", "2505", "2506", "2507"]),
    "Hemiplegia":  (2, ["3341", "342", "343", "3440", "3441", "3442", "3443", "3444", "3445", "3446", "3449"]),
    "Renal":       (2, ["582", "5830", "5831", "5832", "5834", "5836", "5837", "585", "586", "5880", "V420", "V451", "V56"]),
    "MalignNoMet": (2, ["140", "141", "142", "143", "144", "145", "146", "147", "148", "149",
                        "150", "151", "152", "153", "154", "155", "156", "157", "158", "159",
                        "160", "161", "162", "163", "164", "165",
                        "170", "171", "172", "174", "175", "176",
                        "179", "180", "181", "182", "183", "184", "185", "186", "187", "188", "189",
                        "190", "191", "192", "193", "194", "195",
                        "200", "201", "202", "203", "204", "205", "206", "207", "208",
                        "2386"]),
    "SevereLiver": (3, ["4560", "4561", "4562", "5722", "5723", "5724", "5725", "5726", "5727", "5728"]),
    # weight: 6
    "Metastatic":  (6, ["196", "197", "198", "199"]),
    "AIDS":        (6, ["042", "043", "044"]),
}


def _icd9_match(code, prefixes):
    code = str(code).strip()
    return any(code.startswith(p) for p in prefixes)


def compute_charlson(diagnoses):
    """返回以 hadm_id 为索引的 charlson_index Series。"""
    scores = {}
    for hadm_id, grp in diagnoses.groupby("hadm_id"):
        codes = grp["icd9_code"].dropna().astype(str).tolist()
        total = 0
        for disease, (weight, prefixes) in CHARLSON_MAP.items():
            if any(_icd9_match(c, prefixes) for c in codes):
                total += weight
        scores[hadm_id] = total

    return pd.Series(scores, name="charlson_index").reset_index().rename(
        columns={"index": "hadm_id"}
    )


# ════════════════════════════════════════════════════════
# Track A 专属：入院后 24h Lab 值
# ════════════════════════════════════════════════════════

LAB_ITEMS = {
    50912: "creatinine",
    51006: "bun",
    51301: "wbc",
    51265: "platelets",
    51222: "hemoglobin",
    50885: "bilirubin",
    50813: "lactate",
}

# 合理范围过滤（防止异常值污染统计）
LAB_CLIP = {
    50912: (0, 30),    # creatinine mg/dL
    51006: (0, 300),   # BUN mg/dL
    51301: (0, 200),   # WBC K/uL
    51265: (0, 2000),  # Platelets K/uL
    51222: (0, 25),    # Hemoglobin g/dL
    50885: (0, 50),    # Bilirubin mg/dL
    50813: (0, 30),    # Lactate mmol/L
}


def build_lab_features(labevents, admissions):
    """
    提取每次住院（hadm_id）admittime 后 24h 内的 Lab 值，
    汇总为 min/max/mean，缺失值用全局中位数填充。
    """
    # 只保留目标 itemid
    lab = labevents[labevents["itemid"].isin(LAB_ITEMS)].copy()
    lab["charttime"] = pd.to_datetime(lab["charttime"])

    # 关联 admittime
    adm = admissions[["hadm_id", "admittime"]].copy()
    adm["admittime"] = pd.to_datetime(adm["admittime"])
    lab = lab.merge(adm, on="hadm_id", how="left")

    # 24h 时间窗口过滤
    lab["hours_from_admit"] = (lab["charttime"] - lab["admittime"]).dt.total_seconds() / 3600
    lab = lab[(lab["hours_from_admit"] >= 0) & (lab["hours_from_admit"] <= 24)]

    # 异常值 clip
    for itemid, (lo, hi) in LAB_CLIP.items():
        mask = lab["itemid"] == itemid
        lab.loc[mask, "valuenum"] = lab.loc[mask, "valuenum"].clip(lo, hi)

    # 按 hadm_id + itemid 聚合
    agg = (
        lab.groupby(["hadm_id", "itemid"])["valuenum"]
        .agg(["min", "max", "mean"])
        .reset_index()
    )

    # 宽表 pivot
    rows = []
    for itemid, name in LAB_ITEMS.items():
        sub = agg[agg["itemid"] == itemid][["hadm_id", "min", "max", "mean"]].copy()
        sub = sub.rename(columns={
            "min":  f"{name}_min",
            "max":  f"{name}_max",
            "mean": f"{name}_mean",
        })
        rows.append(sub.set_index("hadm_id"))

    lab_wide = pd.concat(rows, axis=1).reset_index()

    # 缺失值：全局中位数填充
    lab_cols = [c for c in lab_wide.columns if c != "hadm_id"]
    medians = lab_wide[lab_cols].median()
    lab_wide[lab_cols] = lab_wide[lab_cols].fillna(medians)

    print(f"  Lab 特征矩阵：{lab_wide.shape}  ({len(lab_cols)} 个特征列)")
    return lab_wide


# ════════════════════════════════════════════════════════
# Track B 专属：ICU 前 24h 生命体征
# ════════════════════════════════════════════════════════

# 同一体征在 CareVue / MetaVision 下的多个 itemid
VITAL_ITEMS = {
    "hr":   ([211, 220045],   (0, 300)),    # 心率
    "sbp":  ([51, 220050],    (0, 300)),    # 收缩压
    "dbp":  ([8368, 220051],  (0, 200)),    # 舒张压
    "spo2": ([646, 220277],   (50, 100)),   # 血氧
    "temp": ([223761, 678],   (25, 45)),    # 体温（°C / °F 混合，后续统一）
    "rr":   ([618, 220210],   (0, 80)),     # 呼吸率
}

ALL_VITAL_ITEMIDS = [iid for ids, _ in VITAL_ITEMS.values() for iid in ids]


def build_vital_features(chartevents, icustays):
    """
    提取每次 ICU 入住（icustay_id）intime 后 24h 内的生命体征，
    汇总为 min/max/mean，缺失值用全局中位数填充。
    """
    ce = chartevents[chartevents["itemid"].isin(ALL_VITAL_ITEMIDS)].copy()
    ce["charttime"] = pd.to_datetime(ce["charttime"])

    icu = icustays[["icustay_id", "hadm_id", "intime"]].copy()
    icu["intime"] = pd.to_datetime(icu["intime"])
    ce = ce.merge(icu, on="icustay_id", how="left")

    # 24h 时间窗口
    ce["hours_from_intime"] = (ce["charttime"] - ce["intime"]).dt.total_seconds() / 3600
    ce = ce[(ce["hours_from_intime"] >= 0) & (ce["hours_from_intime"] <= 24)]

    rows = []
    for name, (itemids, (lo, hi)) in VITAL_ITEMS.items():
        sub = ce[ce["itemid"].isin(itemids)].copy()
        sub["valuenum"] = sub["valuenum"].clip(lo, hi)

        # 体温：CareVue(678) 使用华氏度，转摄氏度
        if name == "temp":
            mask_f = sub["itemid"] == 678
            sub.loc[mask_f, "valuenum"] = (sub.loc[mask_f, "valuenum"] - 32) * 5 / 9
            sub["valuenum"] = sub["valuenum"].clip(25, 45)  # 再 clip 一次

        agg = (
            sub.groupby("icustay_id")["valuenum"]
            .agg(**{
                f"{name}_min":  "min",
                f"{name}_max":  "max",
                f"{name}_mean": "mean",
            })
            .reset_index()
        )
        rows.append(agg.set_index("icustay_id"))

    vital_wide = pd.concat(rows, axis=1).reset_index()

    # 关联 hadm_id
    vital_wide = vital_wide.merge(icu[["icustay_id", "hadm_id"]], on="icustay_id", how="left")

    # 缺失值：全局中位数填充
    vital_cols = [c for c in vital_wide.columns if c not in ["icustay_id", "hadm_id"]]
    medians = vital_wide[vital_cols].median()
    vital_wide[vital_cols] = vital_wide[vital_cols].fillna(medians)

    print(f"  体征特征矩阵：{vital_wide.shape}  ({len(vital_cols)} 个特征列)")
    return vital_wide


# ════════════════════════════════════════════════════════
# Track B 专属：诊断特征
# ════════════════════════════════════════════════════════

def build_diagnosis_features(diagnoses):
    """每次住院的诊断数量。"""
    diag_count = (
        diagnoses.groupby("hadm_id")["icd9_code"]
        .count()
        .reset_index()
        .rename(columns={"icd9_code": "diag_count"})
    )
    return diag_count


# ════════════════════════════════════════════════════════
# 主函数
# ════════════════════════════════════════════════════════

def main():
    print("=" * 55)
    print("阶段 2-FE：特征工程")
    print("=" * 55)

    # ── 加载原始表 ──────────────────────────────────────
    print("\n[1] 加载原始数据 ...")
    patients   = read("PATIENTS.csv",
                      cols=["subject_id", "gender", "dob"],
                      dates=["dob"])
    admissions = read("ADMISSIONS.csv",
                      cols=["subject_id", "hadm_id", "admittime", "dischtime",
                            "admission_type", "hospital_expire_flag"],
                      dates=["admittime", "dischtime"])
    icustays   = read("ICUSTAYS.csv",
                      cols=["subject_id", "hadm_id", "icustay_id",
                            "first_careunit", "intime", "outtime", "los"],
                      dates=["intime", "outtime"])
    diagnoses  = read("DIAGNOSES_ICD.csv",
                      cols=["subject_id", "hadm_id", "icd9_code"])
    labevents  = read("LABEVENTS.csv",
                      cols=["subject_id", "hadm_id", "itemid", "charttime", "valuenum"],
                      dates=["charttime"])

    admissions["hospital_expire_flag"] = admissions["hospital_expire_flag"].fillna(0).astype(int)

    print(f"  PATIENTS:    {len(patients):,}")
    print(f"  ADMISSIONS:  {len(admissions):,}")
    print(f"  ICUSTAYS:    {len(icustays):,}")
    print(f"  DIAGNOSES:   {len(diagnoses):,}")
    print(f"  LABEVENTS:   {len(labevents):,}")

    # ── 共用特征 ────────────────────────────────────────
    print("\n[2] 构建人口统计学特征 ...")
    demo = build_demographics(patients, admissions)
    print(f"  人口统计特征：{demo.shape}")

    print("\n[3] 计算 Charlson 合并症指数 ...")
    charlson = compute_charlson(diagnoses)
    print(f"  Charlson 指数：{charlson.shape}  均值={charlson['charlson_index'].mean():.2f}")

    # ── Track A ─────────────────────────────────────────
    print("\n[4] 构建 Track A Lab 特征（入院后 24h）...")
    lab_feats = build_lab_features(labevents, admissions)

    print("\n[5] 组装 Track A 特征矩阵 ...")
    track_a = (
        demo
        .merge(charlson, on="hadm_id", how="left")
        .merge(lab_feats,  on="hadm_id", how="left")
    )
    # Charlson 缺失 → 0（该次住院无诊断记录）
    track_a["charlson_index"] = track_a["charlson_index"].fillna(0)

    # 目标列
    target_a = "hospital_expire_flag"
    feature_cols_a = [c for c in track_a.columns
                      if c not in ["hadm_id", "subject_id", "admittime",
                                   "dischtime", target_a]]
    print(f"  Track A 特征数：{len(feature_cols_a)}  样本数：{len(track_a)}")
    print(f"  死亡率：{track_a[target_a].mean()*100:.1f}%")
    track_a.to_csv("features_track_a.csv", index=False)
    print("  ✓ 已保存 features_track_a.csv")

    # ── Track B ─────────────────────────────────────────
    print("\n[6] 加载并过滤 CHARTEVENTS（体征 itemid）...")
    chartevents = load_chartevents_filtered(ALL_VITAL_ITEMIDS)

    print("\n[7] 构建 Track B 体征特征（ICU 前 24h）...")
    vital_feats = build_vital_features(chartevents, icustays)

    print("\n[8] 构建诊断数量特征 ...")
    diag_feats = build_diagnosis_features(diagnoses)

    print("\n[9] 组装 Track B 特征矩阵 ...")
    # Track B 以 icustay 为粒度
    icu_base = icustays[["icustay_id", "hadm_id", "subject_id", "los"]].copy()

    track_b = (
        icu_base
        .merge(demo.drop(columns=["subject_id", "dischtime", "hospital_expire_flag"]),
               on="hadm_id", how="left")
        .merge(charlson,    on="hadm_id", how="left")
        .merge(diag_feats,  on="hadm_id", how="left")
        .merge(vital_feats.drop(columns=["hadm_id"]), on="icustay_id", how="left")
    )
    track_b["charlson_index"] = track_b["charlson_index"].fillna(0)
    track_b["diag_count"]     = track_b["diag_count"].fillna(0)

    target_b = "los"
    feature_cols_b = [c for c in track_b.columns
                      if c not in ["icustay_id", "hadm_id", "subject_id",
                                   "admittime", target_b]]
    print(f"  Track B 特征数：{len(feature_cols_b)}  样本数：{len(track_b)}")
    print(f"  LOS 中位数：{track_b[target_b].median():.2f} 天")
    track_b.to_csv("features_track_b.csv", index=False)
    print("  ✓ 已保存 features_track_b.csv")

    print("\n✅ 阶段 2-FE 完成！")
    print(f"   Track A：features_track_a.csv  ({len(track_a)} 行 × {len(feature_cols_a)+1} 列含标签)")
    print(f"   Track B：features_track_b.csv  ({len(track_b)} 行 × {len(feature_cols_b)+1} 列含标签)")

    return track_a, track_b, feature_cols_a, feature_cols_b


track_a, track_b, feat_cols_a, feat_cols_b = main()

阶段 2-FE：特征工程

[1] 加载原始数据 ...
  PATIENTS:    100
  ADMISSIONS:  129
  ICUSTAYS:    136
  DIAGNOSES:   1,761
  LABEVENTS:   76,074

[2] 构建人口统计学特征 ...
    年龄：正常 120 条，>89岁哨兵填充 9 条（age=91）
  人口统计特征：(129, 11)

[3] 计算 Charlson 合并症指数 ...
  Charlson 指数：(129, 2)  均值=3.52

[4] 构建 Track A Lab 特征（入院后 24h）...
  Lab 特征矩阵：(128, 22)  (21 个特征列)

[5] 组装 Track A 特征矩阵 ...
  Track A 特征数：28  样本数：129
  死亡率：31.0%
  ✓ 已保存 features_track_a.csv

[6] 加载并过滤 CHARTEVENTS（体征 itemid）...
  CHARTEVENTS filtered: 64,588 行

[7] 构建 Track B 体征特征（ICU 前 24h）...
  体征特征矩阵：(132, 20)  (18 个特征列)

[8] 构建诊断数量特征 ...

[9] 组装 Track B 特征矩阵 ...
  Track B 特征数：26  样本数：136
  LOS 中位数：2.11 天
  ✓ 已保存 features_track_b.csv

✅ 阶段 2-FE 完成！
   Track A：features_track_a.csv  (129 行 × 29 列含标签)
   Track B：features_track_b.csv  (136 行 × 27 列含标签)


In [30]:
# Quick sanity check
print("Track A shape:", track_a.shape)
print("Track B shape:", track_b.shape)
print("\nTrack A — mortality rate:", f"{track_a['hospital_expire_flag'].mean()*100:.1f}%")
print("Track B — LOS median:", f"{track_b['los'].median():.2f} days")
print("\nAge imputed count:", track_a['age_imputed'].sum())


Track A shape: (129, 33)
Track B shape: (136, 31)

Track A — mortality rate: 31.0%
Track B — LOS median: 2.11 days

Age imputed count: 9


---
## Stage 2A — Mortality Prediction (Track A)

Output: `results_track_a.pkl`

Models: Logistic Regression · Random Forest · XGBoost  
Key metric: **CV AUROC** (5-fold stratified, more reliable than single test-set AUROC with n=26)

In [31]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
)
from sklearn.pipeline import Pipeline
import xgboost as xgb

# ════════════════════════════════════════════════════════
# 加载数据
# ════════════════════════════════════════════════════════

def load_features():
    df = pd.read_csv("features_track_a.csv")

    TARGET = "hospital_expire_flag"
    DROP   = ["hadm_id", "subject_id", "admittime", "dischtime", TARGET]
    feat_cols = [c for c in df.columns if c not in DROP]

    X = df[feat_cols].copy()
    y = df[TARGET].copy()

    # 填充残余缺失（保险起见）
    X = X.fillna(X.median(numeric_only=True))

    print(f"  样本数：{len(X)}  特征数：{len(feat_cols)}")
    print(f"  死亡率：{y.mean()*100:.1f}%  (死亡={y.sum()}, 存活={len(y)-y.sum()})")
    print(f"  特征列：{feat_cols}")
    return X, y, feat_cols


# ════════════════════════════════════════════════════════
# 模型训练
# ════════════════════════════════════════════════════════

def train_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"\n  Train: {len(X_train)}  Test: {len(X_test)}")

    models = {
        "逻辑回归": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42,
                C=0.1,
            )),
        ]),
        "随机森林": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            max_depth=6,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
            eval_metric="auc",
            random_state=42,
            verbosity=0,
        ),
    }

    results = {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, model in models.items():
        print(f"\n  ── {name} ──")
        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        auroc = roc_auc_score(y_test, y_prob)
        ap    = average_precision_score(y_test, y_prob)
        fpr, tpr, thresholds = roc_curve(y_test, y_prob)
        cm    = confusion_matrix(y_test, y_pred)
        cr    = classification_report(y_test, y_pred,
                                      target_names=["存活", "死亡"],
                                      output_dict=True)

        # 5-fold CV AUROC
        cv_scores = cross_val_score(model, X, y, cv=cv,
                                    scoring="roc_auc", n_jobs=-1)

        print(f"    AUROC:      {auroc:.3f}")
        print(f"    AP:         {ap:.3f}")
        print(f"    CV AUROC:   {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
        print(f"    死亡 Recall: {cr['死亡']['recall']:.3f}")
        print(f"    死亡 F1:     {cr['死亡']['f1-score']:.3f}")
        print(f"    混淆矩阵:\n{cm}")

        # 特征重要性
        feat_imp = _get_feature_importance(model, name, X.columns.tolist())

        results[name] = {
            "model":       model,
            "auroc":       auroc,
            "ap":          ap,
            "cv_auroc":    cv_scores,
            "fpr":         fpr,
            "tpr":         tpr,
            "cm":          cm,
            "report":      cr,
            "feat_imp":    feat_imp,
            "y_test":      y_test.values,
            "y_prob":      y_prob,
            "y_pred":      y_pred,
            "feat_cols":   X.columns.tolist(),
        }

    return results, X_test, y_test


def _get_feature_importance(model, name, feat_cols):
    if name == "逻辑回归":
        coef = model.named_steps["clf"].coef_[0]
        imp  = pd.Series(np.abs(coef), index=feat_cols)
    elif name == "随机森林":
        imp = pd.Series(model.feature_importances_, index=feat_cols)
    elif name == "XGBoost":
        imp = pd.Series(model.feature_importances_, index=feat_cols)
    else:
        return None
    return imp.sort_values(ascending=False)


# ════════════════════════════════════════════════════════
# 主函数
# ════════════════════════════════════════════════════════

def main():
    print("=" * 55)
    print("阶段 2A：Track A 死亡率预测建模")
    print("=" * 55)

    print("\n[1] 加载特征矩阵 ...")
    X, y, feat_cols = load_features()

    print("\n[2] 训练模型 ...")
    results, X_test, y_test = train_models(X, y)

    print("\n[3] 保存结果 ...")
    with open("results_track_a.pkl", "wb") as f:
        pickle.dump(results, f)
    print("  ✓ 已保存 results_track_a.pkl")

    best = max(results.items(), key=lambda x: x[1]["auroc"])
    print(f"\n✅ 2A 完成！最佳模型：{best[0]}  AUROC={best[1]['auroc']:.3f}")
    return results


results_a = main()

阶段 2A：Track A 死亡率预测建模

[1] 加载特征矩阵 ...
  样本数：129  特征数：28
  死亡率：31.0%  (死亡=40, 存活=89)
  特征列：['age', 'age_imputed', 'gender_M', 'adm_EMERGENCY', 'adm_ELECTIVE', 'adm_URGENT', 'charlson_index', 'creatinine_min', 'creatinine_max', 'creatinine_mean', 'bun_min', 'bun_max', 'bun_mean', 'wbc_min', 'wbc_max', 'wbc_mean', 'platelets_min', 'platelets_max', 'platelets_mean', 'hemoglobin_min', 'hemoglobin_max', 'hemoglobin_mean', 'bilirubin_min', 'bilirubin_max', 'bilirubin_mean', 'lactate_min', 'lactate_max', 'lactate_mean']

[2] 训练模型 ...

  Train: 103  Test: 26

  ── 逻辑回归 ──
    AUROC:      0.542
    AP:         0.461
    CV AUROC:   0.590 ± 0.074
    死亡 Recall: 0.500
    死亡 F1:     0.444
    混淆矩阵:
[[12  6]
 [ 4  4]]

  ── 随机森林 ──
    AUROC:      0.438
    AP:         0.485
    CV AUROC:   0.598 ± 0.073
    死亡 Recall: 0.375
    死亡 F1:     0.429
    混淆矩阵:
[[15  3]
 [ 5  3]]

  ── XGBoost ──
    AUROC:      0.590
    AP:         0.410
    CV AUROC:   0.609 ± 0.050
    死亡 Recall: 0.500
    死亡 F1:    

---
## Stage 2B — ICU LOS Prediction (Track B)

Output: `results_track_b.pkl`

Models: Baseline (mean) · XGBoost · LightGBM  
Target: `log1p(los)` → predict → `expm1` back to days

In [32]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
import xgboost as xgb
import lightgbm as lgb


# ════════════════════════════════════════════════════════
# 加载数据
# ════════════════════════════════════════════════════════

def load_features():
    df = pd.read_csv("features_track_b.csv")

    TARGET = "los"
    DROP   = ["icustay_id", "hadm_id", "subject_id", "admittime", TARGET]
    feat_cols = [c for c in df.columns if c not in DROP]

    # 过滤异常 LOS（负数或 0）
    df = df[df[TARGET] > 0].copy()

    X = df[feat_cols].copy()
    y = df[TARGET].copy()

    X = X.fillna(X.median(numeric_only=True))

    print(f"  样本数：{len(X)}  特征数：{len(feat_cols)}")
    print(f"  LOS 分布：中位数={y.median():.2f}天  均值={y.mean():.2f}天  最大={y.max():.2f}天")
    print(f"  特征列：{feat_cols}")
    return X, y, feat_cols


# ════════════════════════════════════════════════════════
# 评估函数
# ════════════════════════════════════════════════════════

def evaluate(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"    MAE:  {mae:.3f} 天")
    print(f"    RMSE: {rmse:.3f} 天")
    print(f"    R²:   {r2:.3f}")
    return {"name": name, "mae": mae, "rmse": rmse, "r2": r2}


# ════════════════════════════════════════════════════════
# 模型训练
# ════════════════════════════════════════════════════════

def train_models(X, y):
    # log1p 变换目标变量（右偏分布）
    y_log = np.log1p(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_log, test_size=0.2, random_state=42
    )
    y_test_orig = np.expm1(y_test)  # 反变换，用于最终评估

    print(f"\n  Train: {len(X_train)}  Test: {len(X_test)}")

    results = {}
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    # ── Baseline：均值预测 ──────────────────────────────
    print("\n  ── Baseline（均值预测）──")
    dummy = DummyRegressor(strategy="mean")
    dummy.fit(X_train, y_train)
    y_pred_dummy = np.expm1(dummy.predict(X_test))
    metrics_dummy = evaluate(y_test_orig, y_pred_dummy, "Baseline")
    results["Baseline"] = {
        **metrics_dummy,
        "y_pred": y_pred_dummy,
        "feat_imp": None,
    }

    # ── XGBoost ────────────────────────────────────────
    print("\n  ── XGBoost ──")
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbosity=0,
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False,
    )
    y_pred_xgb = np.expm1(xgb_model.predict(X_test))
    metrics_xgb = evaluate(y_test_orig, y_pred_xgb, "XGBoost")

    cv_scores_xgb = -cross_val_score(
        xgb_model, X, y_log, cv=cv,
        scoring="neg_mean_absolute_error", n_jobs=-1
    )
    print(f"    CV MAE: {cv_scores_xgb.mean():.3f} ± {cv_scores_xgb.std():.3f}")

    feat_imp_xgb = pd.Series(
        xgb_model.feature_importances_,
        index=X.columns.tolist()
    ).sort_values(ascending=False)

    results["XGBoost"] = {
        **metrics_xgb,
        "cv_mae":  cv_scores_xgb,
        "y_pred":  y_pred_xgb,
        "feat_imp": feat_imp_xgb,
    }

    # ── LightGBM ───────────────────────────────────────
    print("\n  ── LightGBM ──")
    lgb_model = lgb.LGBMRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )
    lgb_model.fit(X_train, y_train)
    y_pred_lgb = np.expm1(lgb_model.predict(X_test))
    metrics_lgb = evaluate(y_test_orig, y_pred_lgb, "LightGBM")

    cv_scores_lgb = -cross_val_score(
        lgb_model, X, y_log, cv=cv,
        scoring="neg_mean_absolute_error", n_jobs=-1
    )
    print(f"    CV MAE: {cv_scores_lgb.mean():.3f} ± {cv_scores_lgb.std():.3f}")

    feat_imp_lgb = pd.Series(
        lgb_model.feature_importances_,
        index=X.columns.tolist()
    ).sort_values(ascending=False)

    results["LightGBM"] = {
        **metrics_lgb,
        "cv_mae":  cv_scores_lgb,
        "y_pred":  y_pred_lgb,
        "feat_imp": feat_imp_lgb,
    }

    # 共用数据存入每个结果
    for r in results.values():
        r["y_test_orig"] = y_test_orig.values
        r["feat_cols"]   = X.columns.tolist()

    return results, X_test, y_test_orig


# ════════════════════════════════════════════════════════
# 主函数
# ════════════════════════════════════════════════════════

def main():
    print("=" * 55)
    print("阶段 2B：Track B ICU LOS 预测建模")
    print("=" * 55)

    print("\n[1] 加载特征矩阵 ...")
    X, y, feat_cols = load_features()

    print("\n[2] 训练模型 ...")
    results, X_test, y_test_orig = train_models(X, y)

    print("\n[3] 汇总对比 ...")
    print(f"\n  {'模型':<12} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
    print("  " + "-" * 38)
    for name, r in results.items():
        print(f"  {name:<12} {r['mae']:>8.3f} {r['rmse']:>8.3f} {r['r2']:>8.3f}")

    print("\n[4] 保存结果 ...")
    with open("results_track_b.pkl", "wb") as f:
        pickle.dump(results, f)
    print("  ✓ 已保存 results_track_b.pkl")

    best = min(
        [(k, v) for k, v in results.items() if k != "Baseline"],
        key=lambda x: x[1]["mae"]
    )
    print(f"\n✅ 2B 完成！最佳模型：{best[0]}  MAE={best[1]['mae']:.3f}天")
    return results


results_b = main()

阶段 2B：Track B ICU LOS 预测建模

[1] 加载特征矩阵 ...
  样本数：136  特征数：26
  LOS 分布：中位数=2.11天  均值=4.45天  最大=35.41天
  特征列：['age', 'age_imputed', 'gender_M', 'adm_EMERGENCY', 'adm_ELECTIVE', 'adm_URGENT', 'charlson_index', 'diag_count', 'hr_min', 'hr_max', 'hr_mean', 'sbp_min', 'sbp_max', 'sbp_mean', 'dbp_min', 'dbp_max', 'dbp_mean', 'spo2_min', 'spo2_max', 'spo2_mean', 'temp_min', 'temp_max', 'temp_mean', 'rr_min', 'rr_max', 'rr_mean']

[2] 训练模型 ...

  Train: 108  Test: 28

  ── Baseline（均值预测）──
    MAE:  2.235 天
    RMSE: 3.651 天
    R²:   -0.014

  ── XGBoost ──
    MAE:  2.411 天
    RMSE: 3.486 天
    R²:   0.075
    CV MAE: 0.530 ± 0.057

  ── LightGBM ──
    MAE:  2.626 天
    RMSE: 3.986 天
    R²:   -0.209
    CV MAE: 0.564 ± 0.047

[3] 汇总对比 ...

  模型                MAE     RMSE       R²
  --------------------------------------
  Baseline        2.235    3.651   -0.014
  XGBoost         2.411    3.486    0.075
  LightGBM        2.626    3.986   -0.209

[4] 保存结果 ...
  ✓ 已保存 results_track_b.pkl

✅ 

---
## Stage 2-DB — Modeling Dashboard

Output: `modeling_dashboard.html` — open in browser after running.

In [33]:

"""
阶段 2-DB：Level 2 建模结果 HTML 仪表板
输入：results_track_a.pkl, results_track_b.pkl
输出：modeling_dashboard.html
运行：python stage2_dashboard.py
"""

import pickle
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots


# ════════════════════════════════════════════════════════
# Track A 图表
# ════════════════════════════════════════════════════════

def fig_roc_curves(results_a):
    """ROC 曲线对比（三个模型）"""
    COLORS = {"逻辑回归": "#5B8DEF", "随机森林": "#4ECDC4", "XGBoost": "#FF6B6B"}
    fig = go.Figure()

    for name, r in results_a.items():
        fig.add_trace(go.Scatter(
            x=r["fpr"], y=r["tpr"],
            mode="lines",
            name=f"{name} (AUROC={r['auroc']:.3f})",
            line=dict(color=COLORS.get(name, "#888"), width=2),
            hovertemplate=f"{name}<br>FPR=%{{x:.3f}}<br>TPR=%{{y:.3f}}<extra></extra>",
        ))

    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        name="Random",
        line=dict(color="#555", width=1, dash="dash"),
        showlegend=True,
    ))
    fig.update_layout(
        title=dict(text="ROC 曲线对比", font=dict(size=15, color="#e8eaf0")),
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        xaxis=dict(range=[0, 1]),
        yaxis=dict(range=[0, 1.05]),
        legend=dict(x=0.55, y=0.1, bgcolor="rgba(0,0,0,0)"),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(26,29,46,0.6)",
        font=dict(color="#e8eaf0"),
        margin=dict(t=50, b=40, l=20, r=20),
        height=360,
    )
    return fig


def fig_confusion_matrix(results_a):
    """最佳模型的混淆矩阵（XGBoost）"""
    # 优先取 XGBoost，否则取第一个
    name = "XGBoost" if "XGBoost" in results_a else list(results_a.keys())[0]
    r  = results_a[name]
    cm = r["cm"]

    labels = ["存活", "死亡"]
    fig = go.Figure(go.Heatmap(
        z=cm,
        x=["预测：存活", "预测：死亡"],
        y=["实际：存活", "实际：死亡"],
        colorscale=[[0, "#1a1d2e"], [1, "#FF6B6B"]],
        showscale=False,
        text=cm,
        texttemplate="%{text}",
        textfont=dict(size=22, color="#fff"),
        hovertemplate="实际：%{y}<br>预测：%{x}<br>数量：%{z}<extra></extra>",
    ))
    fig.update_layout(
        title=dict(text=f"混淆矩阵（{name}）", font=dict(size=15, color="#e8eaf0")),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(26,29,46,0.6)",
        font=dict(color="#e8eaf0"),
        margin=dict(t=50, b=40, l=20, r=20),
        height=300,
    )
    return fig


def fig_feature_importance_a(results_a):
    """Track A 特征重要性（Top 15）"""
    name = "XGBoost" if "XGBoost" in results_a else list(results_a.keys())[0]
    imp  = results_a[name]["feat_imp"].head(15).sort_values()

    # 特征名映射（更可读）
    NAME_MAP = {
        "age": "年龄", "age_imputed": "年龄缺失标记", "gender_M": "性别(男)",
        "adm_EMERGENCY": "急诊入院", "adm_ELECTIVE": "择期入院", "adm_URGENT": "紧急入院",
        "charlson_index": "Charlson指数",
        "creatinine_min": "肌酐_min", "creatinine_max": "肌酐_max", "creatinine_mean": "肌酐_mean",
        "bun_min": "BUN_min", "bun_max": "BUN_max", "bun_mean": "BUN_mean",
        "wbc_min": "WBC_min", "wbc_max": "WBC_max", "wbc_mean": "WBC_mean",
        "platelets_min": "血小板_min", "platelets_max": "血小板_max", "platelets_mean": "血小板_mean",
        "hemoglobin_min": "血红蛋白_min", "hemoglobin_max": "血红蛋白_max", "hemoglobin_mean": "血红蛋白_mean",
        "bilirubin_min": "胆红素_min", "bilirubin_max": "胆红素_max", "bilirubin_mean": "胆红素_mean",
        "lactate_min": "乳酸_min", "lactate_max": "乳酸_max", "lactate_mean": "乳酸_mean",
    }
    labels = [NAME_MAP.get(f, f) for f in imp.index]

    fig = go.Figure(go.Bar(
        x=imp.values,
        y=labels,
        orientation="h",
        marker=dict(
            color=imp.values,
            colorscale=[[0, "#2a3a6a"], [1, "#5B8DEF"]],
            showscale=False,
        ),
        hovertemplate="%{y}：%{x:.4f}<extra></extra>",
    ))
    fig.update_layout(
        title=dict(text=f"特征重要性 Top 15（{name}）", font=dict(size=15, color="#e8eaf0")),
        xaxis_title="重要性分数",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(26,29,46,0.6)",
        font=dict(color="#e8eaf0"),
        yaxis=dict(automargin=True, tickfont=dict(size=11)),
        margin=dict(t=50, b=40, l=20, r=20),
        height=400,
    )
    return fig


def make_metric_cards_a(results_a):
    """Track A 指标卡片 HTML"""
    cards = []
    for name, r in results_a.items():
        cr = r["report"]
        recall_death = cr.get("死亡", {}).get("recall", 0)
        f1_death     = cr.get("死亡", {}).get("f1-score", 0)
        cv_mean      = r.get("cv_auroc", np.array([r["auroc"]])).mean()
        cards.append(f"""
        <div class="metric-group">
          <div class="model-name">{name}</div>
          <div class="metrics-row">
            <div class="metric-card">
              <div class="metric-val">{r['auroc']:.3f}</div>
              <div class="metric-label">AUROC</div>
            </div>
            <div class="metric-card">
              <div class="metric-val">{cv_mean:.3f}</div>
              <div class="metric-label">CV AUROC</div>
            </div>
            <div class="metric-card highlight">
              <div class="metric-val">{recall_death:.3f}</div>
              <div class="metric-label">死亡 Recall</div>
            </div>
            <div class="metric-card">
              <div class="metric-val">{f1_death:.3f}</div>
              <div class="metric-label">死亡 F1</div>
            </div>
          </div>
        </div>""")
    return "\n".join(cards)


# ════════════════════════════════════════════════════════
# Track B 图表
# ════════════════════════════════════════════════════════

def fig_pred_vs_actual(results_b):
    """预测值 vs 实际值散点图"""
    COLORS = {"Baseline": "#888", "XGBoost": "#FF6B6B", "LightGBM": "#4ECDC4"}
    fig = go.Figure()

    y_max = 0
    for name, r in results_b.items():
        y_true = r["y_test_orig"]
        y_pred = r["y_pred"]
        y_max  = max(y_max, y_true.max(), y_pred.max())

        fig.add_trace(go.Scatter(
            x=y_true, y=y_pred,
            mode="markers",
            name=f"{name} (MAE={r['mae']:.2f}d)",
            marker=dict(color=COLORS.get(name, "#888"), size=7, opacity=0.65),
            hovertemplate=f"{name}<br>实际：%{{x:.2f}}天<br>预测：%{{y:.2f}}天<extra></extra>",
        ))

    # 对角线（完美预测线）
    fig.add_trace(go.Scatter(
        x=[0, y_max], y=[0, y_max],
        mode="lines",
        name="完美预测",
        line=dict(color="#FFD93D", width=1.5, dash="dash"),
    ))
    fig.update_layout(
        title=dict(text="预测 vs 实际 LOS", font=dict(size=15, color="#e8eaf0")),
        xaxis_title="实际 LOS（天）",
        yaxis_title="预测 LOS（天）",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(26,29,46,0.6)",
        font=dict(color="#e8eaf0"),
        legend=dict(x=0.02, y=0.95, bgcolor="rgba(0,0,0,0)"),
        margin=dict(t=50, b=40, l=20, r=20),
        height=360,
    )
    return fig


def fig_error_distribution(results_b):
    """误差分布直方图"""
    COLORS = {"Baseline": "#888", "XGBoost": "#FF6B6B", "LightGBM": "#4ECDC4"}
    fig = go.Figure()

    for name, r in results_b.items():
        errors = r["y_pred"] - r["y_test_orig"]
        fig.add_trace(go.Histogram(
            x=errors,
            name=name,
            opacity=0.65,
            nbinsx=30,
            marker_color=COLORS.get(name, "#888"),
            hovertemplate=f"{name}<br>误差：%{{x:.2f}}天<br>频次：%{{y}}<extra></extra>",
        ))

    fig.add_vline(x=0, line_dash="dash", line_color="#FFD93D", line_width=1.5,
                  annotation_text="零误差", annotation_font_color="#FFD93D")
    fig.update_layout(
        title=dict(text="预测误差分布（预测 − 实际）", font=dict(size=15, color="#e8eaf0")),
        xaxis_title="误差（天）",
        yaxis_title="频次",
        barmode="overlay",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(26,29,46,0.6)",
        font=dict(color="#e8eaf0"),
        margin=dict(t=50, b=40, l=20, r=20),
        height=320,
    )
    return fig


def fig_feature_importance_b(results_b):
    """Track B 特征重要性（XGBoost 或 LightGBM Top 15）"""
    name = "LightGBM" if "LightGBM" in results_b else "XGBoost"
    if results_b[name]["feat_imp"] is None:
        return go.Figure()

    imp = results_b[name]["feat_imp"].head(15).sort_values()

    NAME_MAP = {
        "age": "年龄", "age_imputed": "年龄缺失标记", "gender_M": "性别(男)",
        "adm_EMERGENCY": "急诊入院", "adm_ELECTIVE": "择期入院", "adm_URGENT": "紧急入院",
        "charlson_index": "Charlson指数", "diag_count": "诊断数量",
        "hr_min": "心率_min", "hr_max": "心率_max", "hr_mean": "心率_mean",
        "sbp_min": "收缩压_min", "sbp_max": "收缩压_max", "sbp_mean": "收缩压_mean",
        "dbp_min": "舒张压_min", "dbp_max": "舒张压_max", "dbp_mean": "舒张压_mean",
        "spo2_min": "血氧_min", "spo2_max": "血氧_max", "spo2_mean": "血氧_mean",
        "temp_min": "体温_min", "temp_max": "体温_max", "temp_mean": "体温_mean",
        "rr_min": "呼吸率_min", "rr_max": "呼吸率_max", "rr_mean": "呼吸率_mean",
    }
    labels = [NAME_MAP.get(f, f) for f in imp.index]

    fig = go.Figure(go.Bar(
        x=imp.values,
        y=labels,
        orientation="h",
        marker=dict(
            color=imp.values,
            colorscale=[[0, "#1a3a2a"], [1, "#4ECDC4"]],
            showscale=False,
        ),
        hovertemplate="%{y}：%{x:.4f}<extra></extra>",
    ))
    fig.update_layout(
        title=dict(text=f"特征重要性 Top 15（{name}）", font=dict(size=15, color="#e8eaf0")),
        xaxis_title="重要性分数",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(26,29,46,0.6)",
        font=dict(color="#e8eaf0"),
        yaxis=dict(automargin=True, tickfont=dict(size=11)),
        margin=dict(t=50, b=40, l=20, r=20),
        height=400,
    )
    return fig


def make_metric_cards_b(results_b):
    """Track B 指标卡片 HTML"""
    cards = []
    for name, r in results_b.items():
        cv_str = ""
        if "cv_mae" in r:
            cv_str = f"""
            <div class="metric-card">
              <div class="metric-val">{r['cv_mae'].mean():.3f}</div>
              <div class="metric-label">CV MAE</div>
            </div>"""
        r2_str = f"{r['r2']:.3f}" if r["r2"] > -999 else "—"
        cards.append(f"""
        <div class="metric-group">
          <div class="model-name">{name}</div>
          <div class="metrics-row">
            <div class="metric-card highlight">
              <div class="metric-val">{r['mae']:.3f}</div>
              <div class="metric-label">MAE（天）</div>
            </div>
            <div class="metric-card">
              <div class="metric-val">{r['rmse']:.3f}</div>
              <div class="metric-label">RMSE（天）</div>
            </div>
            <div class="metric-card">
              <div class="metric-val">{r2_str}</div>
              <div class="metric-label">R²</div>
            </div>
            {cv_str}
          </div>
        </div>""")
    return "\n".join(cards)


# ════════════════════════════════════════════════════════
# 组装 HTML
# ════════════════════════════════════════════════════════

def build_dashboard(results_a, results_b, output_path="modeling_dashboard.html"):
    def to_div(fig):
        return pio.to_html(fig, full_html=False, include_plotlyjs=False)

    # Track A
    roc_div  = to_div(fig_roc_curves(results_a))
    cm_div   = to_div(fig_confusion_matrix(results_a))
    impa_div = to_div(fig_feature_importance_a(results_a))
    cards_a  = make_metric_cards_a(results_a)

    # Track B
    pva_div  = to_div(fig_pred_vs_actual(results_b))
    err_div  = to_div(fig_error_distribution(results_b))
    impb_div = to_div(fig_feature_importance_b(results_b))
    cards_b  = make_metric_cards_b(results_b)

    html = f"""<!DOCTYPE html>
<html lang="zh-CN">
<head>
  <meta charset="UTF-8">
  <title>MIMIC-III Modeling Dashboard</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{
      font-family: 'Segoe UI', 'PingFang SC', 'Microsoft YaHei', sans-serif;
      background: #0f1117;
      color: #e8eaf0;
      min-height: 100vh;
    }}
    header {{
      padding: 24px 40px 16px;
      border-bottom: 1px solid #2a2d3a;
      background: linear-gradient(135deg, #1a1d2e 0%, #0f1117 100%);
    }}
    header h1 {{ font-size: 20px; font-weight: 700; color: #fff; }}
    header p  {{ font-size: 12px; color: #888; margin-top: 4px; }}

    .track-section {{
      padding: 28px 40px;
      max-width: 1400px;
      margin: 0 auto;
    }}
    .track-title {{
      font-size: 16px;
      font-weight: 700;
      color: #fff;
      padding: 10px 16px;
      border-radius: 8px;
      margin-bottom: 20px;
    }}
    .track-a .track-title {{ background: linear-gradient(90deg, #1e3a6e, #1a1d2e); border-left: 4px solid #5B8DEF; }}
    .track-b .track-title {{ background: linear-gradient(90deg, #1a3a2a, #1a1d2e); border-left: 4px solid #4ECDC4; }}

    /* 指标卡片 */
    .metric-group {{ margin-bottom: 20px; }}
    .model-name {{
      font-size: 13px; font-weight: 600; color: #aaa;
      margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.5px;
    }}
    .metrics-row {{ display: flex; gap: 12px; flex-wrap: wrap; }}
    .metric-card {{
      background: #1a1d2e;
      border: 1px solid #2a2d3a;
      border-radius: 10px;
      padding: 14px 20px;
      min-width: 110px;
      text-align: center;
    }}
    .metric-card.highlight {{ border-color: #FF6B6B; }}
    .metric-val {{ font-size: 22px; font-weight: 700; color: #fff; }}
    .metric-label {{ font-size: 11px; color: #888; margin-top: 4px; }}

    /* 图表网格 */
    .chart-grid-2 {{
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 20px;
      margin-bottom: 20px;
    }}
    .chart-grid-3 {{
      display: grid;
      grid-template-columns: 1fr 1fr 1fr;
      gap: 20px;
      margin-bottom: 20px;
    }}
    .chart-card {{
      background: #1a1d2e;
      border: 1px solid #2a2d3a;
      border-radius: 12px;
      padding: 16px;
      overflow: hidden;
    }}
    .chart-card:hover {{ border-color: #5B8DEF; }}
    .track-b .chart-card:hover {{ border-color: #4ECDC4; }}

    .divider {{
      border: none;
      border-top: 1px solid #2a2d3a;
      margin: 8px 40px;
    }}
  </style>
</head>
<body>

<header>
  <h1>MIMIC-III — Level 2 Modeling Dashboard</h1>
  <p>Track A: 院内死亡率预测（分类）&nbsp;·&nbsp;Track B: ICU 住院时长预测（回归）</p>
</header>

<!-- ════ Track A ════ -->
<div class="track-section track-a">
  <div class="track-title">Track A — 院内死亡率预测（Binary Classification）</div>

  {cards_a}

  <div class="chart-grid-2">
    <div class="chart-card">{roc_div}</div>
    <div class="chart-card">{cm_div}</div>
  </div>
  <div class="chart-card">{impa_div}</div>
</div>

<hr class="divider">

<!-- ════ Track B ════ -->
<div class="track-section track-b">
  <div class="track-title">Track B — ICU 住院时长预测（Regression）</div>

  {cards_b}

  <div class="chart-grid-2">
    <div class="chart-card">{pva_div}</div>
    <div class="chart-card">{err_div}</div>
  </div>
  <div class="chart-card">{impb_div}</div>
</div>

</body>
</html>"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ 仪表板已保存：{output_path}")


# ════════════════════════════════════════════════════════
# 主函数
# ════════════════════════════════════════════════════════

def main():
    print("=" * 55)
    print("阶段 2-DB：建模仪表板生成")
    print("=" * 55)

    print("\n[1] 加载 Track A 结果 ...")
    with open("results_track_a.pkl", "rb") as f:
        results_a = pickle.load(f)
    print(f"  模型：{list(results_a.keys())}")

    print("\n[2] 加载 Track B 结果 ...")
    with open("results_track_b.pkl", "rb") as f:
        results_b = pickle.load(f)
    print(f"  模型：{list(results_b.keys())}")

    print("\n[3] 生成仪表板 ...")
    build_dashboard(results_a, results_b)


main()

阶段 2-DB：建模仪表板生成

[1] 加载 Track A 结果 ...
  模型：['逻辑回归', '随机森林', 'XGBoost']

[2] 加载 Track B 结果 ...
  模型：['Baseline', 'XGBoost', 'LightGBM']

[3] 生成仪表板 ...
✅ 仪表板已保存：modeling_dashboard.html


---
## Stage 3a — eDISH Pharmacovigilance

Output: `edish_dashboard.html`

**eDISH quadrants (FDA standard):**
- 🔴 Hy's Law: ALT > 3× ULN AND Bilirubin > 2× ULN → potential serious hepatotoxicity
- 🟣 Temple's Corollary: ALT ≤ 3× AND Bili > 2× → cholestatic injury
- 🟡 Hepatocellular: ALT > 3× AND Bili ≤ 2× → hepatocellular injury without jaundice
- ⚪ Normal: both within range

**Extensions:** KDIGO renal staging + CTCAE hematological grading

In [34]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import os

# DATA_DIR is set in the Setup cell above
OUTPUT_FILE = "edish_dashboard.html"

# Lab itemids
ITEMID_ALT      = 50861   # ALT (SGPT)        U/L
ITEMID_BILI     = 50885   # Total Bilirubin    mg/dL
ITEMID_CREAT    = 50912   # Creatinine         mg/dL
ITEMID_WBC      = 51301   # WBC                K/uL
ITEMID_HGB      = 51222   # Hemoglobin         g/dL
ITEMID_PLT      = 51265   # Platelets          K/uL

# Upper Limits of Normal
ULN_ALT  = 40.0    # U/L
ULN_BILI = 1.2     # mg/dL

# Hy's Law thresholds (× ULN)
HYS_ALT_X  = 3.0
HYS_BILI_X = 2.0

LAYOUT_BASE = dict(
    paper_bgcolor="#ffffff",
    plot_bgcolor="#f8f9fa",
    font=dict(color="#1a1a2e", family="Arial, sans-serif"),
)


# ════════════════════════════════════════════════════════
# 数据提取
# ════════════════════════════════════════════════════════

def load_labevents():
    path = os.path.join(DATA_DIR, "LABEVENTS.csv")
    df = pd.read_csv(path,
                     usecols=["subject_id", "hadm_id", "itemid", "charttime", "valuenum"],
                     parse_dates=["charttime"],
                     low_memory=False)
    # 排除无效值
    df = df[df["valuenum"].notna() & (df["valuenum"] > 0)]
    return df


def extract_peak(labevents, itemid):
    """按 subject_id 取某指标的峰值。"""
    sub = labevents[labevents["itemid"] == itemid]
    peak = sub.groupby("subject_id")["valuenum"].max().reset_index()
    peak.columns = ["subject_id", f"peak_{itemid}"]
    return peak


def build_edish_df(labevents):
    alt_peak  = extract_peak(labevents, ITEMID_ALT)
    bili_peak = extract_peak(labevents, ITEMID_BILI)

    df = alt_peak.merge(bili_peak, on="subject_id", how="inner")
    df = df.rename(columns={
        f"peak_{ITEMID_ALT}":  "alt_peak",
        f"peak_{ITEMID_BILI}": "bili_peak",
    })

    # 归一化
    df["alt_uln"]  = df["alt_peak"]  / ULN_ALT
    df["bili_uln"] = df["bili_peak"] / ULN_BILI

    # 象限分类
    def classify(row):
        a = row["alt_uln"] > HYS_ALT_X
        b = row["bili_uln"] > HYS_BILI_X
        if a and b:     return "Hy's Law"
        if b and not a: return "Temple's Corollary"
        if a and not b: return "Hepatocellular"
        return "Normal"

    df["quadrant"] = df.apply(classify, axis=1)
    df["hys_law"]  = df["quadrant"] == "Hy's Law"

    print(f"  患者数（有 ALT + Bili 记录）：{len(df)}")
    print(f"  Hy's Law 患者：{df['hys_law'].sum()}")
    print(df["quadrant"].value_counts().to_string())
    return df


# ════════════════════════════════════════════════════════
# 延伸：KDIGO 肾脏分期（从肌酐）
# ════════════════════════════════════════════════════════

def build_kdigo(labevents):
    """
    简化版 KDIGO：对每位患者，以该患者肌酐最小值作为基线，
    计算峰值肌酐相对基线的倍数和绝对升高量。
    Stage 1: ≥0.3 mg/dL 升高 or 1.5–1.9× baseline
    Stage 2: 2.0–2.9× baseline
    Stage 3: ≥3.0× baseline or ≥4.0 mg/dL
    """
    creat = labevents[labevents["itemid"] == ITEMID_CREAT].copy()
    creat = creat[(creat["valuenum"] > 0) & (creat["valuenum"] < 30)]

    stats = creat.groupby("subject_id")["valuenum"].agg(
        creat_baseline="min",
        creat_peak="max",
    ).reset_index()
    stats["creat_ratio"]  = stats["creat_peak"] / stats["creat_baseline"]
    stats["creat_delta"]  = stats["creat_peak"] - stats["creat_baseline"]

    def kdigo_stage(row):
        r = row["creat_ratio"]
        d = row["creat_delta"]
        p = row["creat_peak"]
        if r >= 3.0 or p >= 4.0:          return 3
        if 2.0 <= r < 3.0:                return 2
        if (1.5 <= r < 2.0) or d >= 0.3:  return 1
        return 0

    stats["kdigo_stage"] = stats.apply(kdigo_stage, axis=1)
    print("\n  KDIGO 分期分布：")
    print(stats["kdigo_stage"].value_counts().sort_index().to_string())
    return stats


# ════════════════════════════════════════════════════════
# 延伸：CTCAE 血液毒性分级
# ════════════════════════════════════════════════════════

def build_ctcae(labevents):
    """
    基于最差值（最低值对 WBC/HGB/PLT）进行 CTCAE 分级。
    WBC:  Grade 3 < 2.0, Grade 4 < 1.0  (K/uL)
    HGB:  Grade 3 < 8.0, Grade 4 危及生命（用 < 6.5 近似）(g/dL)
    PLT:  Grade 3 < 50,  Grade 4 < 25   (K/uL)
    """
    CTCAE_ITEMS = {
        ITEMID_WBC: ("wbc",  [(4, 2.0), (3, 1.0)]),   # (grade, threshold)
        ITEMID_HGB: ("hgb",  [(4, 8.0), (3, 6.5)]),
        ITEMID_PLT: ("plt",  [(4, 50),  (3, 25)]),
    }

    results = []
    for itemid, (name, grades) in CTCAE_ITEMS.items():
        sub = labevents[labevents["itemid"] == itemid].copy()
        sub = sub[sub["valuenum"] > 0]
        nadir = sub.groupby("subject_id")["valuenum"].min().reset_index()
        nadir.columns = ["subject_id", f"{name}_nadir"]

        def grade(val):
            for g, thresh in grades:
                if val < thresh: return g
            return 0 if val > 0 else np.nan

        nadir[f"{name}_ctcae"] = nadir[f"{name}_nadir"].apply(grade)
        results.append(nadir.set_index("subject_id"))

    ctcae_df = pd.concat(results, axis=1).reset_index()
    print("\n  CTCAE 分级汇总（最高分级人数）：")
    for col in [c for c in ctcae_df.columns if "_ctcae" in c]:
        dist = ctcae_df[col].value_counts().sort_index()
        print(f"    {col}: {dist.to_dict()}")
    return ctcae_df


# ════════════════════════════════════════════════════════
# 图表
# ════════════════════════════════════════════════════════

def fig_edish(edish_df, kdigo_df, ctcae_df):
    """主 eDISH 散点图 + 象限标注。"""

    # 合并 KDIGO 和 CTCAE 用于 hover
    df = edish_df.merge(kdigo_df[["subject_id","kdigo_stage","creat_peak","creat_ratio"]],
                        on="subject_id", how="left")
    df = df.merge(ctcae_df, on="subject_id", how="left")

    # 颜色分组
    QUAD_STYLE = {
        "Normal":              dict(color="#94A3B8", symbol="circle",       size=9,  opacity=0.65),
        "Hepatocellular":      dict(color="#F59E0B", symbol="diamond",      size=11, opacity=0.80),
        "Temple's Corollary":  dict(color="#8B5CF6", symbol="square",       size=11, opacity=0.80),
        "Hy's Law":            dict(color="#EF4444", symbol="star",         size=16, opacity=1.00),
    }

    fig = go.Figure()

    for quad, style in QUAD_STYLE.items():
        sub = df[df["quadrant"] == quad]
        if len(sub) == 0:
            continue

        hover = (
            "<b>Subject %{customdata[0]}</b><br>"
            "Peak ALT: %{customdata[1]:.1f} U/L (%{x:.2f}× ULN)<br>"
            "Peak Bili: %{customdata[2]:.2f} mg/dL (%{y:.2f}× ULN)<br>"
            "KDIGO Stage: %{customdata[3]}<br>"
            "WBC CTCAE: %{customdata[4]}<br>"
            "HGB CTCAE: %{customdata[5]}<br>"
            "PLT CTCAE: %{customdata[6]}"
            "<extra></extra>"
        )
        customdata = sub[[
            "subject_id","alt_peak","bili_peak",
            "kdigo_stage","wbc_ctcae","hgb_ctcae","plt_ctcae"
        ]].values

        fig.add_trace(go.Scatter(
            x=sub["alt_uln"],
            y=sub["bili_uln"],
            mode="markers",
            name=f"{quad} (n={len(sub)})",
            marker=dict(
                color=style["color"],
                symbol=style["symbol"],
                size=style["size"],
                opacity=style["opacity"],
                line=dict(color="#ffffff", width=0.8),
            ),
            customdata=customdata,
            hovertemplate=hover,
        ))

    # 参考线
    ref_lines = [
        (True,  1.0, "#94A3B8", "dash",      "ALT = 1× ULN"),
        (True,  3.0, "#EF4444", "dashdot",   "ALT = 3× ULN (Hy's threshold)"),
        (False, 1.0, "#94A3B8", "dash",      "Bili = 1× ULN"),
        (False, 2.0, "#EF4444", "dashdot",   "Bili = 2× ULN (Hy's threshold)"),
    ]
    for is_vertical, val, color, dash, label in ref_lines:
        if is_vertical:
            fig.add_vline(x=val, line_color=color, line_dash=dash,
                          line_width=1.2, opacity=0.6)
        else:
            fig.add_hline(y=val, line_color=color, line_dash=dash,
                          line_width=1.2, opacity=0.6)

    # 象限标签
    quad_labels = [
        (0.3,  3.5,  "Temple's Corollary",  "#8B5CF6"),
        (4.0,  3.5,  "Hy's Law",            "#EF4444"),
        (0.3,  0.3,  "Normal",              "#64748B"),
        (4.0,  0.3,  "Hepatocellular",      "#F59E0B"),
    ]
    for x, y, text, color in quad_labels:
        fig.add_annotation(
            x=np.log10(x), y=np.log10(y),
            text=f"<b>{text}</b>",
            showarrow=False,
            font=dict(size=11, color=color),
            xref="x", yref="y",
            bgcolor="rgba(255,255,255,0.7)",
            borderpad=3,
        )

    # 坐标轴：对数刻度，手动设置 tickvals
    tick_vals  = [0.1, 0.3, 1, 3, 10, 30, 100]
    tick_texts = ["0.1×", "0.3×", "1×", "3×", "10×", "30×", "100×"]

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text="eDISH Plot — Evaluation of Drug-Induced Serious Hepatotoxicity",
            font=dict(size=17, color="#1a1a2e"), x=0, xanchor="left"
        ),
        xaxis=dict(
            title="Peak ALT / ULN  (ULN = 40 U/L)",
            type="log",
            tickvals=tick_vals,
            ticktext=tick_texts,
            showgrid=True, gridcolor="#e5e7eb",
            title_font=dict(size=13),
            range=[np.log10(0.05), np.log10(200)],
        ),
        yaxis=dict(
            title="Peak Total Bilirubin / ULN  (ULN = 1.2 mg/dL)",
            type="log",
            tickvals=tick_vals,
            ticktext=tick_texts,
            showgrid=True, gridcolor="#e5e7eb",
            title_font=dict(size=13),
            range=[np.log10(0.05), np.log10(50)],
        ),
        legend=dict(
            x=1.01, y=1, xanchor="left",
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#e2e8f0", borderwidth=1,
            font=dict(size=11),
        ),
        margin=dict(t=60, b=60, l=70, r=180),
        height=560,
        hovermode="closest",
    )
    return fig


def fig_kdigo_bar(kdigo_df):
    """KDIGO 分期柱状图。"""
    counts = kdigo_df["kdigo_stage"].value_counts().sort_index()
    labels = {0: "No AKI", 1: "Stage 1", 2: "Stage 2", 3: "Stage 3"}
    colors = {0: "#94A3B8", 1: "#FCD34D", 2: "#F97316", 3: "#EF4444"}

    x = [labels.get(i, str(i)) for i in counts.index]
    c = [colors.get(i, "#888") for i in counts.index]

    fig = go.Figure(go.Bar(
        x=x, y=counts.values,
        marker_color=c,
        marker_line=dict(color="#ffffff", width=1),
        text=counts.values, textposition="outside",
        hovertemplate="<b>%{x}</b><br>Patients: %{y}<extra></extra>",
    ))
    # X 轴标签补充分期标准（用 <br> 换行，Plotly 支持 HTML 标签）
    x_labels = {
        "No AKI":  "No AKI<br>(< 1.5× baseline, Δ < 0.3 mg/dL)",
        "Stage 1": "Stage 1<br>(1.5–1.9× or Δ ≥ 0.3 mg/dL)",
        "Stage 2": "Stage 2<br>(2.0–2.9× baseline)",
        "Stage 3": "Stage 3<br>(≥ 3× or ≥ 4.0 mg/dL)",
    }
    x_display = [x_labels.get(v, v) for v in x]

    fig.update_traces(x=x_display)
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="Renal Safety — KDIGO AKI Staging (from Creatinine)",
                   font=dict(size=15, color="#1a1a2e"), x=0, xanchor="left"),
        yaxis=dict(title="Number of Patients", showgrid=True, gridcolor="#e5e7eb",
                   range=[0, counts.max() * 1.3], dtick=1,
                   title_font=dict(size=12)),
        xaxis=dict(title="KDIGO Stage (creatinine-based criteria)",
                   title_font=dict(size=12),
                   tickfont=dict(size=10)),
        margin=dict(t=50, b=80, l=60, r=20),
        height=360,
        showlegend=False,
    )
    return fig


def fig_ctcae_bar(ctcae_df):
    """CTCAE 血液毒性分级堆叠柱状图。"""
    metrics = [
        ("wbc_ctcae", "WBC",        "#2563EB"),
        ("hgb_ctcae", "Hemoglobin", "#10B981"),
        ("plt_ctcae", "Platelets",  "#F59E0B"),
    ]
    grade_labels = {0: "G0 (Normal)", 1: "G1", 2: "G2", 3: "G3", 4: "G4"}
    grade_colors = {
        0: "#E2E8F0", 1: "#FEF9C3", 2: "#FDE68A", 3: "#F97316", 4: "#EF4444"
    }

    all_grades = sorted(ctcae_df[[m[0] for m in metrics]].stack().dropna().unique())
    fig = go.Figure()

    for grade in all_grades:
        counts = []
        for col, label, _ in metrics:
            counts.append((ctcae_df[col] == grade).sum())
        fig.add_trace(go.Bar(
            name=grade_labels.get(int(grade), f"G{int(grade)}"),
            x=[m[1] for m in metrics],
            y=counts,
            marker_color=grade_colors.get(int(grade), "#888"),
            marker_line=dict(color="#ffffff", width=0.8),
            hovertemplate="<b>%{x}</b><br>Grade %{name}<br>Patients: %{y}<extra></extra>",
        ))

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="Hematological Toxicity — CTCAE Grading",
                   font=dict(size=15, color="#1a1a2e"), x=0, xanchor="left"),
        barmode="stack",
        yaxis=dict(title="Number of Patients", showgrid=True, gridcolor="#e5e7eb"),
        xaxis=dict(title="Lab Marker"),
        legend=dict(title="CTCAE Grade", font=dict(size=11)),
        margin=dict(t=50, b=50, l=60, r=120),
        height=340,
    )
    return fig


# ════════════════════════════════════════════════════════
# 组装 HTML
# ════════════════════════════════════════════════════════

def build_dashboard(fig_main, fig_kdigo, fig_ctcae, edish_df):
    n_hys = int(edish_df["hys_law"].sum())
    n_total = len(edish_df)
    hys_ids = edish_df[edish_df["hys_law"]]["subject_id"].tolist()
    hys_list = ", ".join(str(i) for i in sorted(hys_ids)) if hys_ids else "None"

    def to_div(fig):
        return pio.to_html(fig, full_html=False, include_plotlyjs=False)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>MIMIC-III eDISH Dashboard</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{
      font-family: 'Segoe UI', Arial, sans-serif;
      background: #f1f5f9;
      color: #1a1a2e;
      min-height: 100vh;
    }}
    header {{
      background: #ffffff;
      border-bottom: 2px solid #e2e8f0;
      padding: 24px 40px 18px;
      box-shadow: 0 1px 4px rgba(0,0,0,0.06);
    }}
    header h1 {{ font-size: 21px; font-weight: 700; color: #1e293b; }}
    header p  {{ font-size: 13px; color: #64748b; margin-top: 4px; }}

    .container {{ max-width: 1400px; margin: 0 auto; padding: 28px 40px; }}

    .summary-bar {{
      display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 24px;
    }}
    .stat-card {{
      background: #ffffff; border: 1px solid #e2e8f0;
      border-radius: 10px; padding: 16px 24px;
      box-shadow: 0 1px 4px rgba(0,0,0,0.05);
      min-width: 160px;
    }}
    .stat-card.danger {{ border-left: 4px solid #EF4444; }}
    .stat-card.warn   {{ border-left: 4px solid #F59E0B; }}
    .stat-card.info   {{ border-left: 4px solid #2563EB; }}
    .stat-val  {{ font-size: 28px; font-weight: 700; color: #1e293b; }}
    .stat-label{{ font-size: 12px; color: #64748b; margin-top: 2px; }}

    .hys-box {{
      background: #FEF2F2; border: 1px solid #FECACA;
      border-radius: 10px; padding: 14px 20px;
      margin-bottom: 24px; font-size: 13px; color: #7F1D1D;
    }}
    .hys-box b {{ color: #DC2626; }}

    .chart-card {{
      background: #ffffff; border: 1px solid #e2e8f0;
      border-radius: 12px; padding: 20px;
      box-shadow: 0 1px 6px rgba(0,0,0,0.06);
      margin-bottom: 20px;
    }}
    .grid-2 {{
      display: grid; grid-template-columns: 1fr 1fr;
      gap: 20px; margin-bottom: 20px;
    }}

    .legend-table {{
      width: 100%; border-collapse: collapse;
      font-size: 13px; margin-top: 8px;
    }}
    .legend-table th {{
      background: #f8fafc; text-align: left;
      padding: 8px 12px; border-bottom: 2px solid #e2e8f0;
      color: #475569; font-weight: 600;
    }}
    .legend-table td {{
      padding: 8px 12px; border-bottom: 1px solid #f1f5f9;
      color: #334155;
    }}
    .dot {{
      display: inline-block; width: 10px; height: 10px;
      border-radius: 50%; margin-right: 6px; vertical-align: middle;
    }}
  </style>
</head>
<body>
<header>
  <h1>MIMIC-III Level 3 — eDISH Pharmacovigilance Dashboard</h1>
  <p>Evaluation of Drug-Induced Serious Hepatotoxicity · Renal Safety (KDIGO) · Hematological Toxicity (CTCAE)</p>
</header>

<div class="container">

  <!-- 统计卡片 -->
  <div class="summary-bar">
    <div class="stat-card info">
      <div class="stat-val">{n_total}</div>
      <div class="stat-label">Patients with ALT + Bili Data</div>
    </div>
    <div class="stat-card danger">
      <div class="stat-val">{n_hys}</div>
      <div class="stat-label">Hy's Law Patients</div>
    </div>
    <div class="stat-card warn">
      <div class="stat-val">{n_hys/n_total*100:.1f}%</div>
      <div class="stat-label">Hy's Law Rate</div>
    </div>
  </div>

  <!-- Hy's Law 患者列表 -->
  <div class="hys-box">
    <b>⚠ Hy's Law Patients (ALT &gt; 3× ULN AND Bili &gt; 2× ULN):</b>
    &nbsp;Subject IDs: {hys_list}
  </div>

  <!-- 主 eDISH 图 -->
  <div class="chart-card">
    {to_div(fig_main)}
  </div>

  <!-- 象限说明表 -->
  <div class="chart-card">
    <b style="font-size:14px;">eDISH Quadrant Definitions</b>
    <table class="legend-table">
      <tr>
        <th>Quadrant</th><th>ALT / ULN</th><th>Bili / ULN</th>
        <th>Interpretation</th>
      </tr>
      <tr>
        <td><span class="dot" style="background:#EF4444"></span><b>Hy's Law</b></td>
        <td>&gt; 3×</td><td>&gt; 2×</td>
        <td>Potential serious hepatotoxicity — highest risk signal</td>
      </tr>
      <tr>
        <td><span class="dot" style="background:#8B5CF6"></span>Temple's Corollary</td>
        <td>≤ 3×</td><td>&gt; 2×</td>
        <td>Predominantly cholestatic — bile duct dysfunction</td>
      </tr>
      <tr>
        <td><span class="dot" style="background:#F59E0B"></span>Hepatocellular</td>
        <td>&gt; 3×</td><td>≤ 2×</td>
        <td>Hepatocellular injury without jaundice</td>
      </tr>
      <tr>
        <td><span class="dot" style="background:#94A3B8"></span>Normal</td>
        <td>≤ 3×</td><td>≤ 2×</td>
        <td>Within normal / non-concerning range</td>
      </tr>
    </table>
  </div>

  <!-- KDIGO + CTCAE -->
  <div class="grid-2">
    <div class="chart-card">{to_div(fig_kdigo)}</div>
    <div class="chart-card">{to_div(fig_ctcae)}</div>
  </div>

</div>
</body>
</html>"""

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ Dashboard saved: {OUTPUT_FILE}")


# ════════════════════════════════════════════════════════
# 主函数
# ════════════════════════════════════════════════════════

def main():
    print("=" * 55)
    print("Stage 3a: eDISH Pharmacovigilance")
    print("=" * 55)

    print("\n[1] Loading LABEVENTS ...")
    labevents = load_labevents()
    print(f"  Rows: {len(labevents):,}")

    print("\n[2] Building eDISH dataset ...")
    edish_df = build_edish_df(labevents)

    print("\n[3] KDIGO renal staging ...")
    kdigo_df = build_kdigo(labevents)

    print("\n[4] CTCAE hematological grading ...")
    ctcae_df = build_ctcae(labevents)

    print("\n[5] Generating charts ...")
    fig_main  = fig_edish(edish_df, kdigo_df, ctcae_df)
    fig_kdigo = fig_kdigo_bar(kdigo_df)
    fig_ctcae = fig_ctcae_bar(ctcae_df)

    print("\n[6] Building dashboard ...")
    build_dashboard(fig_main, fig_kdigo, fig_ctcae, edish_df)

    print(f"\n✅ Done! Open in browser: {OUTPUT_FILE}")


main()

Stage 3a: eDISH Pharmacovigilance

[1] Loading LABEVENTS ...
  Rows: 65,319

[2] Building eDISH dataset ...
  患者数（有 ALT + Bili 记录）：76
  Hy's Law 患者：12
quadrant
Normal                55
Hy's Law              12
Hepatocellular         5
Temple's Corollary     4

[3] KDIGO renal staging ...

  KDIGO 分期分布：
kdigo_stage
0    32
1    17
2    33
3    18

[4] CTCAE hematological grading ...

  CTCAE 分级汇总（最高分级人数）：
    wbc_ctcae: {0: 92, 4: 8}
    hgb_ctcae: {0: 62, 4: 38}
    plt_ctcae: {0: 86, 4: 14}

[5] Generating charts ...

[6] Building dashboard ...
✅ Dashboard saved: edish_dashboard.html

✅ Done! Open in browser: edish_dashboard.html


---
## Stage 3b — Patient Story Analysis

Output: `patient_story.html`

**4 analyses:**
1. **Correlation heatmap** — peak Lab values, Pearson r matrix
2. **Deceased vs Survived** — mean peak values by outcome
3. **Outlier detection** — Z-score top-20, colored by mortality
4. **Individual trajectories** — Creatinine / Lactate / Hemoglobin over ICU stay

In [35]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import plotly.io as pio
import os

# DATA_DIR is set in the Setup cell above
OUTPUT_FILE = "patient_story.html"

LAYOUT_BASE = dict(
    paper_bgcolor="#ffffff",
    plot_bgcolor="#f8f9fa",
    font=dict(color="#1a1a2e", family="Arial, sans-serif"),
)

# 关注的 Lab 指标
LAB_ITEMS = {
    50912: ("Creatinine",  "mg/dL",  (0, 20)),
    50813: ("Lactate",     "mmol/L", (0, 20)),
    51301: ("WBC",         "K/uL",   (0, 100)),
    51222: ("Hemoglobin",  "g/dL",   (0, 20)),
    50885: ("Bilirubin",   "mg/dL",  (0, 30)),
    50861: ("ALT",         "U/L",    (0, 1000)),
}
LAB_NAMES = {v[0]: k for k, v in LAB_ITEMS.items()}


# ════════════════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════════════════

def load_data():
    def read(fname, cols=None, dates=None):
        path = os.path.join(DATA_DIR, fname)
        return pd.read_csv(path, usecols=cols, parse_dates=dates, low_memory=False)

    admissions = read("ADMISSIONS.csv",
        cols=["subject_id","hadm_id","admittime","hospital_expire_flag"],
        dates=["admittime"])
    admissions["hospital_expire_flag"] = admissions["hospital_expire_flag"].fillna(0).astype(int)

    icustays = read("ICUSTAYS.csv",
        cols=["subject_id","hadm_id","icustay_id","intime","outtime","los"],
        dates=["intime","outtime"])

    labevents = read("LABEVENTS.csv",
        cols=["subject_id","hadm_id","itemid","charttime","valuenum"],
        dates=["charttime"])

    # 过滤目标 itemid，清除无效值
    lab = labevents[labevents["itemid"].isin(LAB_ITEMS)].copy()
    lab = lab[lab["valuenum"].notna() & (lab["valuenum"] > 0)]

    # clip 异常值
    for itemid, (name, unit, (lo, hi)) in LAB_ITEMS.items():
        mask = lab["itemid"] == itemid
        lab.loc[mask, "valuenum"] = lab.loc[mask, "valuenum"].clip(lo, hi)

    return admissions, icustays, lab


# ════════════════════════════════════════════════════════
# 图 1：Lab 指标相关性热图
# ════════════════════════════════════════════════════════

def fig_correlation_heatmap(lab, admissions):
    """
    Story: 计算各 Lab 指标峰值之间的 Pearson 相关矩阵。
    乳酸与胆红素同步升高 → 多器官功能障碍信号。
    肌酐与 BUN 高度相关 → 肾功能恶化协同模式。
    """
    # 每位患者每个指标的峰值
    pivot = (
        lab.groupby(["subject_id","itemid"])["valuenum"]
        .max().reset_index()
        .pivot(index="subject_id", columns="itemid", values="valuenum")
    )
    pivot.columns = [LAB_ITEMS[c][0] for c in pivot.columns if c in LAB_ITEMS]
    pivot = pivot.dropna(thresh=3)  # 至少有 3 个指标才纳入

    corr = pivot.corr()
    labels = corr.columns.tolist()
    z = corr.values

    # 自定义颜色：负相关蓝，正相关红
    colorscale = [
        [0.0,  "#1D4ED8"],
        [0.5,  "#ffffff"],
        [1.0,  "#DC2626"],
    ]

    fig = go.Figure(go.Heatmap(
        z=z,
        x=labels, y=labels,
        colorscale=colorscale,
        zmid=0, zmin=-1, zmax=1,
        text=np.round(z, 2),
        texttemplate="%{text:.2f}",
        textfont=dict(size=11),
        hovertemplate="<b>%{x} vs %{y}</b><br>Pearson r = %{z:.3f}<extra></extra>",
        colorbar=dict(title="Pearson r", thickness=14, len=0.8),
    ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text="Lab Marker Correlation Matrix (Peak Values per Patient)",
            font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"
        ),
        xaxis=dict(tickfont=dict(size=11)),
        yaxis=dict(tickfont=dict(size=11), autorange="reversed"),
        margin=dict(t=60, b=60, l=80, r=80),
        height=440,
    )
    return fig, corr


# ════════════════════════════════════════════════════════
# 图 2：死亡 vs 存活组 Lab 均值对比
# ════════════════════════════════════════════════════════

def fig_mortality_lab_comparison(lab, admissions):
    """
    Story: 比较死亡组与存活组各 Lab 指标的峰值均值。
    死亡患者通常在肌酐、乳酸、胆红素上显著更高，
    血红蛋白则更低，直观呈现多器官功能恶化趋势。
    """
    # 每位患者每个指标的峰值
    peak = (
        lab.groupby(["subject_id","itemid"])["valuenum"]
        .max().reset_index()
    )
    peak = peak.merge(
        admissions[["subject_id","hospital_expire_flag"]].drop_duplicates("subject_id"),
        on="subject_id", how="left"
    )

    fig = go.Figure()
    COLORS = {"Survived": "#10B981", "Deceased": "#EF4444"}

    for flag, label in [(0, "Survived"), (1, "Deceased")]:
        sub = peak[peak["hospital_expire_flag"] == flag]
        means = sub.groupby("itemid")["valuenum"].mean()

        x_labels, y_vals = [], []
        for itemid, (name, unit, _) in LAB_ITEMS.items():
            if itemid in means.index:
                x_labels.append(f"{name}<br>({unit})")
                y_vals.append(means[itemid])

        fig.add_trace(go.Bar(
            name=label,
            x=x_labels,
            y=y_vals,
            marker_color=COLORS[label],
            marker_line=dict(color="#ffffff", width=1),
            opacity=0.85,
            hovertemplate=f"<b>{label}</b><br>%{{x}}<br>Mean Peak: %{{y:.2f}}<extra></extra>",
        ))

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text="Mean Peak Lab Values: Deceased vs Survived",
            font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"
        ),
        barmode="group",
        yaxis=dict(title="Mean Peak Value", showgrid=True, gridcolor="#e5e7eb"),
        xaxis=dict(title="Lab Marker"),
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)",
                    bordercolor="#e2e8f0", borderwidth=1),
        margin=dict(t=60, b=70, l=60, r=20),
        height=420,
        bargap=0.25, bargroupgap=0.08,
    )
    return fig


# ════════════════════════════════════════════════════════
# 图 3：异常患者识别（Z-score）
# ════════════════════════════════════════════════════════

def fig_outlier_detection(lab, admissions):
    """
    Story: 用 Z-score 识别各指标极端值患者。
    |Z| > 2.5 的患者往往对应高死亡风险或特殊病种。
    """
    peak = (
        lab.groupby(["subject_id","itemid"])["valuenum"]
        .max().reset_index()
        .pivot(index="subject_id", columns="itemid", values="valuenum")
    )
    peak.columns = [LAB_ITEMS[c][0] for c in peak.columns if c in LAB_ITEMS]

    # Z-score
    zscore = (peak - peak.mean()) / peak.std()
    max_z  = zscore.abs().max(axis=1).sort_values(ascending=False)

    # 取 Top 20 异常患者
    top_outliers = max_z.head(20).reset_index()
    top_outliers.columns = ["subject_id", "max_abs_z"]

    # 合并死亡标签
    mortality = admissions[["subject_id","hospital_expire_flag"]].drop_duplicates("subject_id")
    top_outliers = top_outliers.merge(mortality, on="subject_id", how="left")
    top_outliers["outcome"] = top_outliers["hospital_expire_flag"].map(
        {0: "Survived", 1: "Deceased"})

    # 找出每人的极端指标
    def get_extreme_marker(sid):
        row = zscore.loc[sid] if sid in zscore.index else pd.Series()
        if row.empty: return "—"
        return row.abs().idxmax()

    top_outliers["extreme_marker"] = top_outliers["subject_id"].apply(get_extreme_marker)

    colors = top_outliers["hospital_expire_flag"].map(
        {0: "#10B981", 1: "#EF4444"}).fillna("#94A3B8")

    fig = go.Figure(go.Bar(
        x=top_outliers["subject_id"].astype(str),
        y=top_outliers["max_abs_z"],
        marker_color=colors,
        marker_line=dict(color="#ffffff", width=0.5),
        customdata=top_outliers[["outcome","extreme_marker"]].values,
        hovertemplate=(
            "<b>Subject %{x}</b><br>"
            "Max |Z-score|: %{y:.2f}<br>"
            "Extreme marker: %{customdata[1]}<br>"
            "Outcome: %{customdata[0]}"
            "<extra></extra>"
        ),
    ))
    # 阈值线
    fig.add_hline(y=2.5, line_dash="dash", line_color="#F59E0B", line_width=1.5,
                  annotation_text="|Z| = 2.5 threshold",
                  annotation_font=dict(size=11, color="#F59E0B"),
                  annotation_position="top right")

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text="Top 20 Outlier Patients by Max |Z-score| across Lab Markers",
            font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"
        ),
        xaxis=dict(title="Subject ID", tickfont=dict(size=10), tickangle=-35),
        yaxis=dict(title="Max |Z-score|", showgrid=True, gridcolor="#e5e7eb"),
        margin=dict(t=60, b=80, l=60, r=20),
        height=400,
        showlegend=False,
    )
    return fig, top_outliers


# ════════════════════════════════════════════════════════
# 图 4：个体患者 Lab 时序曲线（死亡 vs 存活各选 2 人）
# ════════════════════════════════════════════════════════

def fig_patient_timeseries(lab, admissions, icustays, top_outliers):
    """
    Story: 选取死亡组和存活组各 2 名代表性患者，
    绘制 ICU 住院期间关键 Lab 指标的时序变化。
    死亡患者通常呈现乳酸/肌酐持续攀升，血红蛋白持续下降的趋势。
    """
    # 选患者：死亡组取最异常的 2 人，存活组取数据最丰富的 2 人
    deceased_ids = top_outliers[top_outliers["hospital_expire_flag"] == 1]["subject_id"].head(2).tolist()
    survived_ids = top_outliers[top_outliers["hospital_expire_flag"] == 0]["subject_id"].head(2).tolist()

    # 如果异常列表里存活不够，补充数据丰富的存活患者
    if len(survived_ids) < 2:
        survived_adm = admissions[admissions["hospital_expire_flag"] == 0]["subject_id"].unique()
        counts = lab[lab["subject_id"].isin(survived_adm)].groupby("subject_id").size()
        extra = counts.nlargest(4).index.tolist()
        for sid in extra:
            if sid not in survived_ids:
                survived_ids.append(sid)
            if len(survived_ids) >= 2:
                break

    selected = {
        "Deceased": deceased_ids[:2],
        "Survived": survived_ids[:2],
    }

    # 要展示的指标
    SHOW_ITEMS = [50912, 50813, 51222]  # Creatinine, Lactate, Hemoglobin
    ITEM_NAMES = {50912: "Creatinine (mg/dL)", 50813: "Lactate (mmol/L)",
                  51222: "Hemoglobin (g/dL)"}

    COLOR_DEAD = ["#EF4444", "#F97316"]
    COLOR_SURV = ["#10B981", "#2563EB"]

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[ITEM_NAMES[i] for i in SHOW_ITEMS],
        horizontal_spacing=0.10,
    )

    icu_merge = icustays[["subject_id","intime"]].drop_duplicates("subject_id")

    for group_label, sids in selected.items():
        colors = COLOR_DEAD if group_label == "Deceased" else COLOR_SURV
        for ci, sid in enumerate(sids):
            # 获取该患者 ICU intime 作为时间零点
            icu_row = icu_merge[icu_merge["subject_id"] == sid]
            if icu_row.empty:
                adm_row = admissions[admissions["subject_id"] == sid]
                t0 = pd.to_datetime(adm_row["admittime"].iloc[0]) if not adm_row.empty else None
            else:
                t0 = pd.to_datetime(icu_row["intime"].iloc[0])

            pat_lab = lab[lab["subject_id"] == sid].copy()
            if t0 is not None:
                pat_lab["hours"] = (
                    pd.to_datetime(pat_lab["charttime"]) - t0
                ).dt.total_seconds() / 3600
            else:
                pat_lab["hours"] = np.nan

            # 只保留 ICU 期间 (-6h ~ 120h)
            pat_lab = pat_lab[(pat_lab["hours"] >= -6) & (pat_lab["hours"] <= 120)]

            show_legend = (ci == 0)
            for col_idx, itemid in enumerate(SHOW_ITEMS, 1):
                sub = pat_lab[pat_lab["itemid"] == itemid].sort_values("hours")
                if len(sub) == 0:
                    continue
                fig.add_trace(go.Scatter(
                    x=sub["hours"],
                    y=sub["valuenum"],
                    mode="lines+markers",
                    name=f"{group_label} – S{sid}",
                    line=dict(color=colors[ci], width=2,
                              dash="solid" if group_label == "Deceased" else "dot"),
                    marker=dict(size=5, color=colors[ci]),
                    legendgroup=f"{group_label}_{sid}",
                    showlegend=(show_legend and col_idx == 1),
                    hovertemplate=(
                        f"<b>Subject {sid} ({group_label})</b><br>"
                        f"Hour: %{{x:.1f}}<br>Value: %{{y:.2f}}<extra></extra>"
                    ),
                ),
                row=1, col=col_idx)

    for col_idx in range(1, 4):
        fig.update_xaxes(title_text="Hours from ICU Admission",
                         showgrid=True, gridcolor="#e5e7eb",
                         title_font=dict(size=11), row=1, col=col_idx)
        fig.update_yaxes(showgrid=True, gridcolor="#e5e7eb", row=1, col=col_idx)

    for ann in fig.layout.annotations:
        ann.font.color = "#1a1a2e"
        ann.font.size  = 13

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text="Individual Patient Lab Trajectories During ICU Stay",
            font=dict(size=16, color="#1a1a2e"), x=0, xanchor="left"
        ),
        legend=dict(
            x=1.01, y=1, xanchor="left",
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#e2e8f0", borderwidth=1,
            font=dict(size=11),
            title=dict(text="Patient (Solid=Deceased, Dotted=Survived)"),
        ),
        margin=dict(t=70, b=60, l=60, r=200),
        height=420,
    )
    return fig, selected


# ════════════════════════════════════════════════════════
# 组装 HTML
# ════════════════════════════════════════════════════════

def build_dashboard(fig_corr, corr, fig_mort, fig_outlier, top_outliers,
                    fig_ts, selected_patients):

    def to_div(fig):
        return pio.to_html(fig, full_html=False, include_plotlyjs=False)

    # 关键发现汇总
    # 找相关性最高的非对角线对
    corr_pairs = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            corr_pairs.append((cols[i], cols[j], corr.iloc[i, j]))
    corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    top_pair = corr_pairs[0] if corr_pairs else ("—", "—", 0)

    n_outliers_deceased = int((top_outliers["hospital_expire_flag"] == 1).sum())
    deceased_selected = selected_patients.get("Deceased", [])
    survived_selected = selected_patients.get("Survived", [])

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>MIMIC-III Patient Story Analysis</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{
      font-family: 'Segoe UI', Arial, sans-serif;
      background: #f1f5f9; color: #1a1a2e; min-height: 100vh;
    }}
    header {{
      background: #ffffff; border-bottom: 2px solid #e2e8f0;
      padding: 24px 40px 18px;
      box-shadow: 0 1px 4px rgba(0,0,0,0.06);
    }}
    header h1 {{ font-size: 21px; font-weight: 700; color: #1e293b; }}
    header p  {{ font-size: 13px; color: #64748b; margin-top: 4px; }}

    .container {{ max-width: 1400px; margin: 0 auto; padding: 28px 40px; }}

    .insight-grid {{
      display: grid; grid-template-columns: repeat(3, 1fr);
      gap: 16px; margin-bottom: 24px;
    }}
    .insight-card {{
      background: #ffffff; border: 1px solid #e2e8f0;
      border-radius: 10px; padding: 16px 20px;
      border-left: 4px solid #2563EB;
      box-shadow: 0 1px 4px rgba(0,0,0,0.05);
    }}
    .insight-card.green {{ border-left-color: #10B981; }}
    .insight-card.red   {{ border-left-color: #EF4444; }}
    .insight-title {{ font-size: 12px; font-weight: 600; color: #64748b;
                      text-transform: uppercase; letter-spacing: 0.5px; }}
    .insight-body  {{ font-size: 13px; color: #334155; margin-top: 6px; line-height: 1.5; }}

    .chart-card {{
      background: #ffffff; border: 1px solid #e2e8f0;
      border-radius: 12px; padding: 20px;
      box-shadow: 0 1px 6px rgba(0,0,0,0.06);
      margin-bottom: 20px;
    }}
    .story-note {{
      background: #F0FDF4; border: 1px solid #BBF7D0;
      border-radius: 8px; padding: 12px 16px;
      font-size: 13px; color: #14532D;
      margin-top: 10px; line-height: 1.6;
    }}
    .story-note.warn {{
      background: #FEF9C3; border-color: #FDE68A; color: #713F12;
    }}
    .story-note.info {{
      background: #EFF6FF; border-color: #BFDBFE; color: #1E3A8A;
    }}
    .grid-2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
  </style>
</head>
<body>
<header>
  <h1>MIMIC-III Level 3 — Patient Lab Story Analysis</h1>
  <p>Connecting multi-dimensional lab data to tell the story of ICU patients · Patterns, outliers, and individual trajectories</p>
</header>

<div class="container">

  <!-- Key Insights -->
  <div class="insight-grid">
    <div class="insight-card">
      <div class="insight-title">🔗 Strongest Lab Correlation</div>
      <div class="insight-body">
        <b>{top_pair[0]}</b> &amp; <b>{top_pair[1]}</b><br>
        Pearson r = <b>{top_pair[2]:.2f}</b> — co-elevation suggests shared pathophysiology
      </div>
    </div>
    <div class="insight-card red">
      <div class="insight-title">⚠ Outlier Patients (Deceased)</div>
      <div class="insight-body">
        <b>{n_outliers_deceased}</b> of top-20 outlier patients had in-hospital death,
        suggesting extreme lab values are strong mortality signals
      </div>
    </div>
    <div class="insight-card green">
      <div class="insight-title">📈 Trajectory Patients Selected</div>
      <div class="insight-body">
        Deceased: Subject(s) <b>{', '.join(str(s) for s in deceased_selected)}</b><br>
        Survived: Subject(s) <b>{', '.join(str(s) for s in survived_selected)}</b>
      </div>
    </div>
  </div>

  <!-- 图 1: Correlation Heatmap -->
  <div class="chart-card">
    {to_div(fig_corr)}
    <div class="story-note info">
      📖 <b>What this tells us:</b> Strong positive correlations (dark red) between markers
      like Lactate &amp; Bilirubin or Creatinine &amp; BUN suggest simultaneous multi-organ
      dysfunction — a hallmark of sepsis and critical illness. Negative correlations with
      Hemoglobin reflect anemia commonly co-occurring with organ failure.
    </div>
  </div>

  <!-- 图 2: Deceased vs Survived -->
  <div class="chart-card">
    {to_div(fig_mort)}
    <div class="story-note warn">
      📖 <b>What this tells us:</b> Deceased patients consistently show higher peak values
      for Creatinine, Lactate, Bilirubin and ALT — markers of renal, circulatory, and
      hepatic failure. Lower Hemoglobin in the deceased group reflects anemia and possible
      hemorrhage. These differences align with known ICU mortality predictors.
    </div>
  </div>

  <!-- 图 3 + 图 4: Outliers + Trajectories -->
  <div class="chart-card">
    {to_div(fig_outlier)}
    <div class="story-note warn">
      📖 <b>What this tells us:</b> Patients with the highest Z-scores (most extreme values
      relative to the cohort) are disproportionately from the deceased group (red bars).
      Hover over each bar to see which marker drove the outlier status — Lactate and
      Creatinine are common culprits.
    </div>
  </div>

  <div class="chart-card">
    {to_div(fig_ts)}
    <div class="story-note">
      📖 <b>What this tells us:</b> Deceased patients (solid lines) typically show a
      progressive rise in Creatinine and Lactate over the ICU stay, indicating worsening
      renal function and tissue hypoperfusion. Hemoglobin trends downward. Survived
      patients (dotted lines) tend to show stabilization or improvement — a visual
      signature of recovery vs. deterioration.
    </div>
  </div>

</div>
</body>
</html>"""

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ Dashboard saved: {OUTPUT_FILE}")


# ════════════════════════════════════════════════════════
# 主函数
# ════════════════════════════════════════════════════════

def main():
    print("=" * 55)
    print("Stage 3b: Patient Story Analysis")
    print("=" * 55)

    print("\n[1] Loading data ...")
    admissions, icustays, lab = load_data()
    print(f"  Lab events (filtered): {len(lab):,}")

    print("\n[2] Correlation heatmap ...")
    fig_corr, corr = fig_correlation_heatmap(lab, admissions)

    print("\n[3] Deceased vs Survived lab comparison ...")
    fig_mort = fig_mortality_lab_comparison(lab, admissions)

    print("\n[4] Outlier detection (Z-score) ...")
    fig_outlier, top_outliers = fig_outlier_detection(lab, admissions)
    print(f"  Top outliers: {len(top_outliers)}")
    print(f"  Deceased in top-20: {(top_outliers['hospital_expire_flag']==1).sum()}")

    print("\n[5] Individual patient trajectories ...")
    fig_ts, selected = fig_patient_timeseries(lab, admissions, icustays, top_outliers)
    print(f"  Deceased patients: {selected.get('Deceased', [])}")
    print(f"  Survived patients: {selected.get('Survived', [])}")

    print("\n[6] Building dashboard ...")
    build_dashboard(fig_corr, corr, fig_mort, fig_outlier, top_outliers,
                    fig_ts, selected)

    print(f"\n✅ Done! Open in browser: {OUTPUT_FILE}")


main()

Stage 3b: Patient Story Analysis

[1] Loading data ...
  Lab events (filtered): 8,033

[2] Correlation heatmap ...

[3] Deceased vs Survived lab comparison ...

[4] Outlier detection (Z-score) ...
  Top outliers: 20
  Deceased in top-20: 10

[5] Individual patient trajectories ...
  Deceased patients: [10126, 41983]
  Survived patients: [44212, 10061]

[6] Building dashboard ...
✅ Dashboard saved: patient_story.html

✅ Done! Open in browser: patient_story.html


---
## Output Files Summary

In [36]:
import os
outputs = [
    "eda_dashboard.html",
    "modeling_dashboard.html",
    "edish_dashboard.html",
    "patient_story.html",
    "features_track_a.csv",
    "features_track_b.csv",
    "results_track_a.pkl",
    "results_track_b.pkl",
]
print("Output files:")
for f in outputs:
    exists = "✅" if os.path.exists(f) else "❌ missing"
    size = f"{os.path.getsize(f)/1024:.1f} KB" if os.path.exists(f) else ""
    print(f"  {exists}  {f:<40} {size}")


Output files:
  ✅  eda_dashboard.html                       121.5 KB
  ✅  modeling_dashboard.html                  61.7 KB
  ✅  edish_dashboard.html                     41.0 KB
  ✅  patient_story.html                       50.0 KB
  ✅  features_track_a.csv                     24.3 KB
  ✅  features_track_b.csv                     29.6 KB
  ✅  results_track_a.pkl                      642.8 KB
  ✅  results_track_b.pkl                      3.0 KB
